# **CS Course Project — Local Differential Privacy with RAPPOR**
### *Federated Learning–Style Simulation on the NSDUH 2021–2023 Dataset*

This project investigates the **privacy–utility tradeoff of Local Differential Privacy (LDP)**
in a federated-learning–style setting using the **NSDUH 2021–2023 national survey dataset**.
Each data record is treated as an independent *client* that locally privatizes its categorical
feature using the **RAPPOR mechanism** before any aggregation or learning occurs.

From the server’s perspective, only **noisy binary reports** are ever observed. Global category
statistics are then reconstructed through **regularized statistical decoding**, and supervised
models are trained using only these privatized reconstructions.

On top of RAPPOR, we introduce a **post-processing structural anonymization layer**
based on **label-aware dynamic programming (DP) segmentation**, which reduces domain
granularity to suppress linkage and memorization risk while preserving predictive utility.

---

## **Pipeline Overview**

### 1. Client-Side Privacy (Local Differential Privacy)
Each simulated client applies the full **RAPPOR privatization pipeline**:

- Bloom-filter encoding  
- Permanent randomized response  
- Instantaneous randomized response  

Only **privatized Bloom vectors** are sent to the server.  
**Raw categorical values are never shared.**

---

### 2. Server-Side Statistical Decoding
The server aggregates privatized reports and **decodes category frequencies** using a
ridge-regularized linear inversion of the Bloom + randomized-response process.

This reconstructs a **global categorical distribution** without exposing any individual records.

---

### 3. Structural Anonymization (Label-Aware DP Segmentation)
We apply a **dynamic programming segmentation algorithm** to the decoded distribution,
using:

- decoded category frequency  
- per-category label prevalence (label-aware weighting)

This compresses the domain into contiguous segments, making rare or extreme categories
**indistinguishable by design** and reducing re-identification risk.

---

### 4. Learning Models (Decision Tree & Logistic Regression)
For each drug outcome, three learning pipelines are evaluated:

**Baseline Models**
- Trained on raw (coarsened) categories  
- No privacy applied  

**RAPPOR-Only Models**
- Trained on **synthetic categories sampled from the decoded RAPPOR distribution**
- Server never observes true client features  

**RAPPOR + DP-Seg Models**
- Trained on **DP-segmented RAPPOR features**
- Additional structural anonymization layer applied  

All categorical features are handled via **one-hot encoding**, and trees rely on
scikit-learn’s **standard greedy split selection** (no custom split rule is used).

---

### 5. Privacy & Utility Metrics

We evaluate privacy leakage and learning performance using:

**Distribution Distortion**
- Jensen–Shannon (JS) divergence  

**Structural Leakage**
- Tree depth  
- Leaf count  
- Tree structure similarity (Jaccard over flattened trees)  

**Learning Utility**
- **F1 score (positive class)**  
- Mutual Information (feature–label inference risk proxy)  

> Note: All reported performance values are **F1 scores**, not raw accuracy.

---

### 6. Global RAPPOR Distortion Benchmark
Before any supervised learning, we compute a **global distortion baseline**:

- JS divergence between true and decoded distributions  
- Category match rate (synthetic vs. true)  
- Hamming-style category distortion  

This isolates the **privacy cost of RAPPOR itself**, independent of any model.

---

### 7. Epsilon Sweep (Privacy–Utility Optimization)
Rather than sweeping category resolution, this project performs a **privacy-budget (ε) sweep**:

For each ε value:
- A new RAPPOR decode is generated  
- Global JS distortion is measured  
- All per-drug and group models are retrained  
- F1, distortion, and mutual information are recorded  

A **composite privacy–utility score** is used to identify:
- the best ε **per drug**
- the best ε **overall**
- the best ε **per drug group**

---

## **Goals of the Project**

1. Implement a full **Local Differential Privacy pipeline** using RAPPOR  
2. Quantify how **privacy noise affects supervised learning performance (F1)**  
3. Measure **structural and information-theoretic leakage** in trained models  
4. Study how varying the **privacy budget ε** shifts the privacy–utility frontier  

---

This notebook contains the complete end-to-end implementation, evaluation framework,
and ε-sweep optimization results for all experiments.


## **Environment Setup & Core Utilities**

This section initializes all core Python libraries used throughout the project for:

- numerical computation  
- dataset manipulation  
- local differential privacy simulation  
- supervised learning  
- structural and information-theoretic evaluation  
- experiment logging and reproducibility  

In addition, two global utility functions are defined and reused across all
privacy and learning experiments:

- **Jensen–Shannon (JS) Divergence**  
  Measures distribution-level distortion between the true categorical
  distribution and the RAPPOR-decoded distribution.

- **normalize()**  
  Converts noisy vectors into valid probability distributions after
  clipping and decoding to ensure numerical stability.

These utilities form the computational backbone for evaluating distortion,
privacy leakage, and stability across the full LDP pipeline.


In [1]:
# ============================================================
# IMPORTS & CORE UTILITIES
#
# This block initializes all libraries used throughout the
# privacy, modeling, evaluation, and experiment pipeline.
#
# Includes:
#   - Numerical computation      (NumPy)
#   - Data handling              (Pandas)
#   - Privacy & hashing          (hashlib, math)
#   - Machine learning           (scikit-learn)
#   - Structural evaluation      (tree parsing, JSON)
#   - Experiment logging         (os, joblib)
#   - Visualization              (matplotlib, seaborn)
# ============================================================

# ------------------------------
# Core Numerical & Data Tools
# ------------------------------
import numpy as np
import pandas as pd
from collections import Counter
import math
import hashlib

# ------------------------------
# File I/O & Experiment Logging
# ------------------------------
import os
import json
import joblib
from joblib import dump, load
from pathlib import Path

# ------------------------------
# Visualization (used later)
# ------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------
# Scikit-Learn Core Components
# ------------------------------
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import _tree
from sklearn import tree

from sklearn.linear_model import LogisticRegression

# ------------------------------
# Evaluation Metrics
# ------------------------------

# F1 score (positive class) for learning utility
from sklearn.metrics import f1_score

# Mutual Information for feature–label inference risk
from sklearn.metrics import mutual_info_score

# ------------------------------------------------------
# Utility: Jensen–Shannon (JS) Divergence
#
# Measures global distribution distortion between:
#   - true categorical distribution
#   - RAPPOR-decoded distribution
#
# Used as the primary population-level privacy metric.
# ------------------------------------------------------

def js_divergence(p, q):
    """
    Computes Jensen–Shannon divergence between two probability distributions.

    Properties:
        - symmetric
        - always finite
        - bounded between 0 and 1 (log base 2)

    Parameters:
        p (array-like): True distribution
        q (array-like): Decoded / privatized distribution

    Returns:
        js (float): Jensen–Shannon divergence in bits
    """

    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)

    # enforce valid probability distributions
    p = p / p.sum()
    q = q / q.sum()

    m = 0.5 * (p + q)

    def kl(a, b):
        mask = (a > 0)
        return np.sum(a[mask] * np.log2(a[mask] / b[mask]))

    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


# ------------------------------------------------------
# Utility: Normalize Noisy Vectors
#
# Ensures that any noisy vector produced by:
#   - RAPPOR decoding
#   - DP mechanisms
#
# is a valid probability distribution.
# ------------------------------------------------------

def normalize(v):
    """
    Converts a noisy vector into a valid probability distribution.

    Enforces:
        - non-negativity
        - sum to 1
        - numerical stability after DP noise

    Parameters:
        v (array-like): Noisy vector

    Returns:
        v_norm (np.array): Valid probability distribution
    """

    v = np.maximum(v, 0)
    s = v.sum()

    return v / s if s > 0 else np.ones_like(v) / len(v)


print("Environment setup & core utilities loaded.")

Environment setup & core utilities loaded.


## **Dataset Loading (NSDUH 2021–2023)**

We load the merged NSDUH 2021–2023 public-use dataset (tab-separated format).
The dataset contains approximately **173,000 respondents** and thousands of
survey variables covering:

- demographics  
- education and income  
- psychological distress  
- substance use  
- health behaviors  

Official NSDUH missing-value codes (`-9, -8, -7`) are converted to `NaN` for
consistent processing.

Each row represents a single simulated **federated client**, whose data will
later be privatized locally using RAPPOR before aggregation.

In [2]:
# ============================================================
# LOAD NSDUH 2021–2023 DATASET
#
# Reads in the National Survey on Drug Use and Health data
# containing anonymous survey responses used as the simulated
# population for federated learning clients.
#
# This is the "RAW" dataset before any:
#   - anonymization
#   - RAPPOR encoding
#   - compression
#   - differential privacy
#
# All privacy protections are applied AFTER this stage.
# ============================================================

DATA_PATH = "NSDUH_2021_2023_Tab.txt"   # file must(!) exist in same directory

df = pd.read_csv(
    DATA_PATH,
    sep="\t",               # NSDUH files are tab-delimited
    low_memory=False,       # avoids pandas type guessing warnings on large files
    na_values=[-9, -8, -7]  # official NSDUH missing-value codes
)

# check check checking....
print("Dataset successfully loaded!")
print("Shape (rows, columns):", df.shape)

Dataset successfully loaded!
Shape (rows, columns): (173808, 2639)


## **Feature Engineering: High-Dimensional Categorical Encoding**

RAPPOR operates on **categorical inputs**, so we transform multiple NSDUH attributes
into a **single high-cardinality categorical feature** that represents each
simulated client.

For each respondent, we construct a 6-dimensional categorical profile using:

- Age group  
- Sex  
- Race / ethnicity  
- Education level  
- Poverty category  
- K6 psychological distress (binned into 3 levels)  

Each row is mapped to a **unique integer category ID**, which represents the
true client-side value that is later privatized using RAPPOR.

To ensure valid encoding:
- Rows with missing values in any required field are filtered using a boolean mask.
- Several NSDUH variables are **coarsened into broader groups** to improve
  privacy and reduce domain sparsity.

This produces a **high-dimensional but privacy-aware categorical feature space**
that serves as the input to the Local Differential Privacy pipeline.


In [3]:
# ============================================================
# FEATURE ENGINEERING: HIGH-DIMENSIONAL CATEGORICAL ENCODING
#
# Combines multiple demographic and psychological attributes
# into a single categorical ID per client.
#
# This ID represents the TRUE client-side value that will later
# be privatized using RAPPOR.
# ============================================================

def make_features_coarsened(df):
    """
    Constructs a coarsened categorical feature representation
    from demographic and mental-health attributes.

    Each row is mapped to a single categorical ID representing:
        (age_grp, sex, race_grp, edu_grp, poverty_grp, distress_bin)

    Compared to the original version, this function:
      - Collapses some categories into broader, more privacy-friendly groups
      - Keeps the same boolean mask logic
      - Still returns a single integer ID per row for compatibility
    """

    # -----------------------
    # Gather base attributes
    # -----------------------
    age        = df["CATAG3"]         # age group (NSDUH-coded)
    sex        = df["IRSEX"]          # biological sex
    race       = df["NEWRACE2"]       # race category
    education  = df["EDUHIGHCAT"]     # highest education level
    poverty    = df["POVERTY3"]       # income-to-poverty ratio

    # -----------------------
    # K6 psychological distress indicators
    # -----------------------
    k6_cols = [
        "DSTRST30", "DSTHOP30", "DSTNRV30",
        "DSTCHR30", "DSTEFF30", "DSTNGD30"
    ]
    k6 = df[k6_cols]

    # -----------------------
    # Validity mask: require all feature fields
    # -----------------------
    base_valid = (
        age.notna() &
        sex.notna() &
        race.notna() &
        education.notna() &
        poverty.notna() &
        k6.notna().all(axis=1)
    )

    mask_features = base_valid.to_numpy()
    df_valid = df.loc[mask_features].copy()

    # -----------------------
    # K6 distress score -> 3 bins 
    # -----------------------
    k6_score = df_valid[k6_cols].fillna(0).sum(axis=1)

    distress_bin = pd.cut(
        k6_score,
        bins=[-1, 4, 12, 100],
        labels=[0, 1, 2]   # 0=low, 1=moderate, 2=high
    )

    distress_bin = distress_bin.astype("float").fillna(0).astype(int)

    # ------------------------------------------------------------------
    # COARSENING STEP: collapse raw NSDUH codes into broader categories
    # ------------------------------------------------------------------

    # Age: broader adult bins
    age_raw = df_valid["CATAG3"].astype(int)
    age_grp = age_raw.map({
        1: 1,        # 12–17 
        2: 2,        # 18–25
        3: 2,        # 26–34 -> merge with 18–25 into "young adults"
        4: 3,        # 35–49
        5: 4,        # 50–64
        6: 4         # 65+   -> merge with 50–64
    }).fillna(2).astype(int)

    # Race: White / Black / Hispanic / Other
    race_raw = df_valid["NEWRACE2"].astype(int)
    race_grp = race_raw.map({
        1: 1,  # Non-Hispanic White
        2: 2,  # Non-Hispanic Black
        3: 3,  # Non-Hispanic Native/Alaska (fold into Other)
        4: 3,  # Non-Hispanic Native Hawaiian/PI (Other)
        5: 3,  # Non-Hispanic Asian (Other)
        6: 3,  # Non-Hispanic Multiple race (Other)
        7: 4   # Hispanic
    }).fillna(3).astype(int)

    # Education: HS or less / some college / college+
    edu_raw = df_valid["EDUHIGHCAT"].astype(int)
    edu_grp = edu_raw.map({
        1: 1,  # <HS or HS or less
        2: 1,
        3: 2,  # some college / AA
        4: 3   # BA or more
    }).fillna(2).astype(int)

    # Poverty: keep POVERTY3 as-is or treat as 3 levels (low/mid/high)
    pov_raw = df_valid["POVERTY3"].astype(int)
    # If POVERTY3 is already 1=poor, 2=near, 3=above, you can just reuse it:
    pov_grp = pov_raw  # or remap if needed

    # Sex: keep as-is (usually 1,2)
    sex_bin = df_valid["IRSEX"].astype(int)

    # -----------------------
    # Combine into feature tuples (now coarsened)
    # -----------------------
    tuples = list(zip(
        age_grp,
        sex_bin,
        race_grp,
        edu_grp,
        pov_grp,
        distress_bin
    ))

    mapping_features = {}
    ids = []
    next_id = 0

    for t in tuples:
        if t not in mapping_features:
            mapping_features[t] = next_id
            next_id += 1
        ids.append(mapping_features[t])

    X_features = np.array(ids, dtype=int)

    return X_features, mapping_features, mask_features


In [4]:
# ============================================================
# RUN FEATURE CONSTRUCTION
# ============================================================

X_features, mapping_features, mask_features = make_features_coarsened(df)

df_filtered = df[mask_features].reset_index(drop=True)
X_filtered  = X_features

X_cat = X_filtered


print("After feature-mask filtering:")
print(" -> df_filtered shape:", df_filtered.shape)
print(" -> X_filtered shape:", X_filtered.shape)
print(" -> Number of unique categories:", len(mapping_features))


K_global = len(mapping_features)  

After feature-mask filtering:
 -> df_filtered shape: (173729, 2639)
 -> X_filtered shape: (173729,)
 -> Number of unique categories: 639


## **Drug Label Construction (Binary Targets)**

For supervised learning, NSDUH drug-use and misuse variables are converted into
**binary classification targets** indicating presence or absence of use.

Label construction follows NSDUH survey conventions:

- **Illicit drugs** (e.g., marijuana, cocaine, heroin, methamphetamine, hallucinogens)  
  -> labeled `1` if usage is reported at least once in the past year.

- **Tobacco and nicotine vaping**  
  -> use NSDUH’s binary coding (`0 = no`, `1 = yes`).

- **Prescription misuse categories**  
  -> labeled `1` if misuse is reported.

Invalid survey responses (`85, 94, 97, 98, 99`) are removed using a boolean mask.

For each outcome, this process produces:

- `y` — binary label vector  
- `mask_y` — validity mask for usable rows  
- `X_cat[mask_y]` — categorical client features aligned with valid labels  

Each drug outcome is trained and evaluated **independently** using the same
privacy and modeling pipeline.


In [5]:
##############################################################
# DRUG LABEL CONSTRUCTION (NSDUH -> Binary Targets)
#
# Converts NSDUH survey drug-use variables into binary
# classification labels:
#   - 1 = use / misuse
#   - 0 = no use
#
# Invalid NSDUH response codes are removed using a mask.
#
# Outputs:
#   - y       : binary labels
#   - mask_y  : validity mask aligned to df_filtered and X_cat
##############################################################

# Mapping from experimental drug names → NSDUH variable codes
DRUG_VARIABLES = {
    # Illicit drug use
    "marijuana": "MRJYR",
    "cocaine": "COCYR",
    "crack": "CRKYR",
    "heroin": "HERYR",
    "methamphetamine": "METHAMYR",
    "inhalants": "INHALYR",
    "hallucinogens": "HALLUCYR",

    # Tobacco & vaping
    "tobacco": "CIGYR",
    "nicotine_vaping": "NICVAPYR",

    # Prescription misuse
    "opioid_misuse": "OPIANYYR",
    "tranquilizer_misuse": "TRQANYYR",
    "sedative_misuse": "SEDANYYR",
    "stimulant_misuse": "STMANYYR",
    "pain_reliever_misuse": "PNRANYYR"
}

# Invalid NSDUH response codes
INVALID_CODES = {85, 94, 97, 98, 99}


def make_drug_labels(df, drug_name):
    """
    Converts NSDUH drug-use variables into binary labels.

    Outputs:
        y (np.array[int])   : 0/1 labels
        mask_y (np.array[bool]) : rows with valid survey responses

    mask_y is aligned to df and directly indexes:
        X_cat_comp[mask_y]
        df_filtered[mask_y]
    """
    if drug_name not in DRUG_VARIABLES:
        raise ValueError(f"Unknown drug '{drug_name}'")

    col = DRUG_VARIABLES[drug_name]
    raw = df[col]

    # Identify which rows have valid data
    mask_y = (~raw.isin(INVALID_CODES) & raw.notna()).to_numpy()

    # Extract valid responses only
    vals = raw[mask_y]

    # ----------------------------------------
    # Interpretation of survey response
    # ----------------------------------------

    # Illicit drugs: numeric count of days -> use if > 0
    if drug_name in [
        "marijuana", "cocaine", "crack", "heroin",
        "methamphetamine", "inhalants", "hallucinogens"
    ]:
        y = (vals.astype(float) > 0).astype(int)

    # Tobacco & vaping: binary yes/no
    elif drug_name in ["tobacco", "nicotine_vaping"]:
        y = (vals.astype(int) == 1).astype(int)

    # Prescription misuse: already binary (1/0)
    elif drug_name in [
        "opioid_misuse", "tranquilizer_misuse",
        "sedative_misuse", "stimulant_misuse",
        "pain_reliever_misuse"
    ]:
        y = (vals.astype(int) == 1).astype(int)

    else:
        # fallback safety: treat nonzero as "use"
        y = (vals.astype(float) > 0).astype(int)

    return y.to_numpy(), mask_y


##############################################################
# GROUP LABEL CONSTRUCTION (Multi-Drug Categories)
#
# Builds binary group labels such as:
#   - illicit drugs
#   - nicotine products
#   - prescription misuse
#
# Group label = 1 if ANY member drug is positive.
##############################################################

def make_group_labels(df, group_name, member_drugs):
    """
    Builds binary group label for drug categories (e.g., illicit, nicotine).

    - Uses the intersection of validity across all member drugs.
    - Group = 1 if ANY member drug is used.
    - Returns (labels, mask) aligned to df.

    Outputs:
        y_group : binary labels for valid rows
        mask_group : boolean mask of valid respondents
    """

    n = df.shape[0]
    valid_mask = np.ones(n, dtype=bool)
    aligned_labels = []

    # Build aligned full-length labels for each member drug
    for drug in member_drugs:
        y, mask_y = make_drug_labels(df, drug)

        full = np.full(n, np.nan)
        full[mask_y] = y

        valid_mask &= ~np.isnan(full)
        aligned_labels.append(full)

   # Compute group label on rows valid for all drugs
    group_y = np.zeros(n, dtype=int)
    for full in aligned_labels:
        group_y[valid_mask] |= (full[valid_mask] == 1).astype(int)

    return group_y[valid_mask], valid_mask


## **RAPPOR Client-Side Encoding (Local Differential Privacy)**

This section implements the full **RAPPOR mechanism** for enforcing
**Local Differential Privacy (LDP)** at the client level.

Each simulated client applies RAPPOR *locally* before sending any information
to the server. The server never observes raw categorical values.

### Encoding Pipeline

1. **Bloom Filter Encoding**  
   Each category is mapped to a fixed-length binary vector using multiple
   independent hash functions.

2. **Permanent Randomized Response**  
   Introduces persistent client-side noise to prevent long-term tracking
   across repeated reports.

3. **Instantaneous Randomized Response**  
   Adds fresh per-report randomness, enforcing formal ε-local differential
   privacy guarantees.

The function `rappor_client_reports()` simulates one fully privatized RAPPOR
report per user. All downstream learning and analysis operate **only on these
privatized reports**.


In [6]:
##############################################################
# RAPPOR PARAMETERs SETUP!!
#
# These parameters control the privacy–utility behavior of
# the RAPPOR mechanism. The local privacy budget ε depends
# only on (p, q) from instantaneous randomized response.
##############################################################

def rappor_epsilon(p, q):
    """
    Compute the local differential privacy (epsilon) of RAPPOR
    from instantaneous randomized response parameters.

    ε = ln( q(1 - p) / ( p(1 - q) ) )
    """
    return math.log((q * (1 - p)) / (p * (1 - q)))


def q_from_epsilon(epsilon, p=0.5):
    """
    Solve for q given target epsilon and fixed p.

    From:
        ε = ln( q(1 - p) / (p(1 - q)) )

    we get:
        q = p * exp(ε) / (p * exp(ε) - p + 1)
    """
    r = math.exp(epsilon)
    return (p * r) / (p * r - p + 1)

# ------------------------------
# Default RAPPOR Parameters
# ------------------------------

BLOOM_K = 128     # bits in Bloom filter
BLOOM_H = 2       # number of hash functions
F       = 0.5     # permanent RR flip prob
P       = 0.5     # instantaneous RR prob when base bit = 0
Q       = 0.75    # instantaneous RR prob when base bit = 1

# Example privacy level (used for sanity check only)
# EPSILON = rappor_epsilon(P, Q)
# print(f"RAPPOR Local Privacy Budget ε = {EPSILON:.4f}")

In [7]:
##############################################################
# RAPPOR CLIENT-SIDE ENCODING
#
# Implements:
#   1) Bloom filter hashing
#   2) Permanent randomized response
#   3) Instantaneous randomized response
#
# This enforces formal Local Differential Privacy (LDP).
##############################################################

def hash_indices(value_id, k=BLOOM_K, h=BLOOM_H, salt=""):
    '''
    Maps a categorical value ID to Bloom filter positions.

    Uses SHA-256 hashing to simulate multiple independent hash functions.

    Parameters:
        value_id (int):
            Category ID to encode.
        k (int):
            Bloom filter size (number of bits).
        h (int):
            Number of hash functions.
        salt (str):
            Optional salt to prevent cross-domain linkage.

    Return value(s):
        indices (list):
            Bloom filter indices to activate.
    '''

    indices = []

    # hash category value and salt
    base = f"{value_id}:{salt}".encode("utf-8")
    digest = hashlib.sha256(base).digest()  # 256-bit hash

    # carve hash stream into 4-byte chunks → indices
    for i in range(h):
        chunk = digest[4*i:4*(i+1)]
        idx = int.from_bytes(chunk, byteorder="big") % k
        indices.append(idx)

    return indices


def bloom_encode_category(cat_id, k=BLOOM_K, h=BLOOM_H):
    '''
    Encodes a category ID as a Bloom filter bit vector.

    Parameters:
        cat_id (int):
            Category identifier.
        k (int):
            Bloom filter length.
        h (int):
            Number of hashes.

    Return value(s):
        bits (np.array):
            Binary Bloom filter vector.
    '''
    
    bits = np.zeros(k, dtype=int)

    # set hash-selected indices to 1
    for idx in hash_indices(cat_id, k=k, h=h, salt="rappor"):
        bits[idx] = 1

    return bits


def permanent_rr(bloom_bits, f, rng=None):
    '''
    Applies permanent randomized response to Bloom bits.

    Each bit is:
        - replaced with random noise with probability f
        - preserved with probability (1 - f)

    This makes each client representation stable across reports
    but permanently obfuscated.

    Parameters:
        bloom_bits (array-like):
            Clean Bloom vector.
        f (float):
            Noise probability.
        rng (Generator):
            Numpy random generator.

    Return value(s):
        perm_bits (np.array):
            Permanently perturbed Bloom vector.
    '''

    if rng is None:
        rng = np.random.default_rng()

    # decide which bits to randomize
    flip_mask = rng.random(size=bloom_bits.shape) < f

    # generate replacement randomness
    random_bits = rng.integers(0, 2, size=bloom_bits.shape)

    perm_bits = bloom_bits.copy()
    perm_bits[flip_mask] = random_bits[flip_mask]

    return perm_bits


def instantaneous_rr(perm_bits, p, q, rng=None):
    '''
    Applies real-time randomized response.

        If bit = 1 -> return 1 with probability q
        If bit = 0 -> return 1 with probability p

    Parameters:
        perm_bits (array-like):
            Permanently randomized Bloom vector.
        p (float):
            False positive probability.
        q (float):
            True positive probability.

    Return value(s):
        out (np.array):
            Final privatized bit vector.
    '''

    if rng is None:
        rng = np.random.default_rng()

    r = rng.random(size=perm_bits.shape)

    out = np.zeros_like(perm_bits)

    ones  = (perm_bits == 1)
    zeros = ~ones

    # prob bit reporting
    out[ones]  = (r[ones] < q).astype(int)
    out[zeros] = (r[zeros] < p).astype(int)

    return out


def rappor_client_reports(X_cat, k, h, f, p, q, seed=None):
    '''
    Simulates client-side RAPPOR encoding for all users.

    Each value is:
        category -> bloom -> permanent noise -> instantaneous noise

    Parameters:
        X_cat (array-like):
            Category ID per user.
        k, h, f, p, q:
            RAPPOR parameters.
        seed (int):
            RNG seed for reproducibility.

    Return value(s):
        reports (np.array):
            Binary RAPPOR reports, shape (n_users, bloom_size).
    '''

    rng = np.random.default_rng(seed)
    n   = len(X_cat)

    reports = np.zeros((n, k), dtype=int)

    for i, cat_id in enumerate(X_cat):
        bloom = bloom_encode_category(cat_id, k=k, h=h)
        perm  = permanent_rr(bloom, f=f, rng=rng)
        inst  = instantaneous_rr(perm, p=p, q=q, rng=rng)

        reports[i, :] = inst

    return reports


## **Server-Side Aggregation & Statistical Decoding**

After receiving locally privatized RAPPOR reports from all clients, the server
performs **population-level statistical reconstruction** without ever observing
raw client values.

Specifically, the server:

- aggregates Bloom-filter bits across all users  
- constructs a **design matrix** that models the expected noisy response behavior  
- solves a **ridge-regularized linear inverse problem** to recover category statistics  

This decoding process reconstructs **global categorical frequencies** under
Local Differential Privacy (LDP), while preserving formal privacy guarantees.

The output of this stage consists of:

- `est_counts` — estimated number of users per category  
- `est_probs` — estimated category probability distribution  

These decoded distributions represent the **only feature information available to the server**
and are used as the basis for all downstream learning and privacy analysis.


In [8]:
##############################################################
# RAPPOR SERVER-SIDE AGGREGATION & STATISTICAL DECODING
#
# This stage reconstructs global categorical statistics from
# locally privatized Bloom filter reports using ridge-regularized
# linear inversion.
#
# No individual client values are ever recovered.
##############################################################

def rappor_aggregate(reports):
    '''
    Aggregates RAPPOR client reports by summing bits position-wise.

    Parameters:
        reports (np.array):
            Matrix of RAPPOR reports, shape (n_users, bloom_size).

    Return value(s):
        counts (np.array):
            Vector of bit totals, length bloom_size.
        n (int):
            Number of users.
    '''
    n = reports.shape[0]
    counts = reports.sum(axis=0)
    return counts, n


def build_rappor_design_matrix(K, k, h, f, p, q):
    '''
    Constructs the expected response probability matrix A where:

        A[i, j] = P(final_bit[i] = 1 | category j)

    This matrix captures:
        - Bloom filter assignment
        - permanent randomized response
        - instantaneous randomized response

    Parameters:
        K (int):
            Number of categories.
        k (int):
            Bloom size.
        h (int):
            Number of hash functions.
        f, p, q:
            RAPPOR parameters.

    Return value(s):
        A_prob (np.array):
            Expected probability matrix with shape (bloom_size, num_categories).
    '''

    # compute bloom patterns for every category
    bloom_bits = np.zeros((K, k), dtype=int)
    for j in range(K):
        bloom_bits[j, :] = bloom_encode_category(j, k=k, h=h)

    # permanent randomized response
    alpha1 = 1.0 - f / 2.0     # P(perm = 1 | orig = 1)
    beta1  = f / 2.0           # P(perm = 1 | orig = 0)

    # instantaneous randomized response
    prob1 = alpha1 * q + (1.0 - alpha1) * p   # P(final=1 | orig=1)
    prob0 = beta1  * q + (1.0 - beta1)  * p   # P(final=1 | orig=0)

    # build expected response matrix
    A_prob = np.zeros((k, K), dtype=float)
    for j in range(K):
        for i in range(k):
            A_prob[i, j] = prob1 if bloom_bits[j, i] == 1 else prob0

    return A_prob


def rappor_decode(counts, n, K, k, h, f, p, q, l2_reg=1e-2):
    '''
    Estimates category frequencies from aggregated RAPPOR reports.

    Uses ridge-regularized least squares:

        minimize ||A x - y||^2 + l2_reg ||x||^2

    Enforces:
        - non-negativity
        - distribution normalization

    Parameters:
        counts (np.array):
            Sum of reported Bloom bits.
        n (int):
            Total number of users.
        K (int):
            Number of original categories.
        k, h, f, p, q:
            RAPPOR parameters.
        l2_reg (float):
            Regularization strength.

    Return value(s):
        est_counts (np.array):
            Estimated number of users per category.
        est_probs (np.array):
            Estimated probability distribution.
    '''

    # observed bit probabilities
    y_prob = counts / float(n)

    # expected probabilities from encoding model
    A = build_rappor_design_matrix(K, k=k, h=h, f=f, p=p, q=q)

    # -----------------------------
    # Solve linear system via normal equations
    #
    #   (A.T A + l2_reg*I) x = A.T y
    # -----------------------------
    ATA = A.T @ A
    ATy = A.T @ y_prob
    ATA_reg = ATA + l2_reg * np.eye(K)

    x_hat = np.linalg.solve(ATA_reg, ATy)

    # -----------------------------
    # Enforce probability constraints
    # -----------------------------
    x_hat = np.clip(x_hat, 0, None)

    if x_hat.sum() > 0:
        x_hat = x_hat / x_hat.sum()
    else:
        x_hat = np.ones(K) / K  # fallback: uniform

    est_probs = x_hat
    est_counts = est_probs * n

    return est_counts, est_probs


## **Proposed Algorithm: Label-Aware Dynamic Programming (DP) Segmentation**

RAPPOR introduces substantial statistical noise, particularly in **high-cardinality
categorical domains**, which leads to sparsity, unstable learning, and increased
re-identification risk through rare categories.

To mitigate these effects, we introduce a **post-decoding structural anonymization
layer** that compresses the categorical domain *after* RAPPOR decoding.
Because this transformation operates strictly on **already privatized statistics**,
it does **not weaken the Local Differential Privacy (LDP) guarantee**.

---

### **1. Label-Aware Category Scoring**

For each category \( c \), we compute a combined utility score based on:

- the **RAPPOR-estimated category frequency**
- the **category-level label prevalence**

This produces a score that prioritizes categories that are both:
- statistically significant, and
- predictive with respect to the learning target.

---

### **2. Optimal Segmentation via Dynamic Programming**

Categories are sorted by their combined scores and partitioned into **B contiguous
segments** using **dynamic programming** to minimize within-segment variance:

> Objective: Minimize the sum of squared errors (SSE) within each segment.

This produces:
- smoother feature representations  
- reduced sparsity  
- lower linkage and re-identification risk  

---

### **3. Learning on Segmented Features**

The resulting **segment IDs** replace the original categories as model features.
A standard scikit-learn decision tree is then trained using its built-in greedy splitter.

No custom split rule is applied — **segmentation is the only structural modification**.

---

This approach reduces domain granularity while preserving learning utility and remains
formally **compatible with Local Differential Privacy**, since it is applied only as
a deterministic post-processing step on privatized data.


In [9]:
##############################################################
# DYNAMIC PROGRAMMING (DP) SEGMENTATION FOR CATEGORY MERGING
#
# This algorithm partitions categories into B contiguous
# segments by minimizing the within-segment sum of squared
# errors (SSE) over a sorted utility score.
#
# This reduces:
#   - domain granularity
#   - sparsity
#   - linkage and re-identification risk
#
# The transformation is applied ONLY after RAPPOR decoding,
# and therefore preserves the Local DP guarantee.
##############################################################

def dp_segment_by_values(values, B):
    '''
    Partitions categories into B contiguous segments using
    dynamic programming to minimize within-segment variance.

    This groups similar-frequency categories together to reduce
    distinguishability between sparse values.

    Parameters:
        values (np.array):
            Estimated probabilities per category (length K).
        B (int):
            Desired number of segments.

    Return value(s):
        segment_id (np.array):
            Maps each category -> assigned segment.
        segments (list):
            Lists of category indices per segment.
    '''

    K = len(values)

    # sort categories by frequency
    order = np.argsort(values)
    v = values[order]

    # --------------------------------------------------
    # Precompute prefix sums for O(1) variance queries
    # --------------------------------------------------

    prefix_sum   = np.zeros(K + 1)
    prefix_sumsq = np.zeros(K + 1)

    for i in range(1, K + 1):
        prefix_sum[i]   = prefix_sum[i-1]   + v[i-1]
        prefix_sumsq[i] = prefix_sumsq[i-1] + v[i-1]**2

    def seg_cost(i, j):
        '''
        Computes sum of squared error of a segment [i..j].

        Used as the objective measure for segmentation quality:
        low variance -> more homogeneous (same) categories.
        '''
        length = j - i + 1
        s  = prefix_sum[j+1]   - prefix_sum[i]
        ss = prefix_sumsq[j+1] - prefix_sumsq[i]

        mean = s / length

        # SSE = the sum of[ (v - mean)^2 ]
        return ss - 2*mean*s + length*(mean**2)

    # --------------------------------------------------
    # Dynamic Programming Setup
    #
    # dp[b][j] = minimum segmentation cost of first j values
    #            into b segments
    # --------------------------------------------------

    INF = float("inf")
    dp   = np.full((B + 1, K + 1), INF)
    prev = np.full((B + 1, K + 1), -1, dtype=int)

    # base case: single segment covering [0..j-1]
    for j in range(1, K + 1):
        dp[1][j] = seg_cost(0, j-1)
        prev[1][j] = 0

    # DP recurrence
    for b in range(2, B + 1):
        for j in range(b, K + 1):
            best_cost = INF
            best_i = -1

            # try all possible previous splits
            for i in range(b-1, j):
                cost = dp[b-1][i] + seg_cost(i, j-1)
                if cost < best_cost:
                    best_cost = cost
                    best_i = i

            dp[b][j] = best_cost
            prev[b][j] = best_i

    # --------------------------------------------------
    # Recover segment boundaries via backtracking
    # --------------------------------------------------

    boundaries = []
    b = B
    j = K

    while b > 0:
        i = prev[b][j]
        boundaries.append((i, j-1))
        j = i
        b -= 1

    boundaries.reverse()

    # --------------------------------------------------
    # Assign segment IDs
    # --------------------------------------------------

    segment_id = np.zeros(K, dtype=int)
    segments = []

    for seg_idx, (start, end) in enumerate(boundaries):
        seg_positions = np.arange(start, end+1)
        seg_cats = order[seg_positions]

        segments.append(seg_cats)

        for c in seg_cats:
            segment_id[c] = seg_idx

    return segment_id, segments


## **Modeling & Evaluation Framework**

From this point forward, the notebook executes the full experimental pipeline for
quantifying **privacy, learning utility, and structural leakage** under Local
Differential Privacy.

For each drug outcome, we compare three supervised learning pipelines:

1. **Baseline Model**  
   Trained on raw categorical features (no privacy, upper-bound utility).

2. **RAPPOR-Only Model**  
   Trained on synthetic categories sampled from **decoded RAPPOR distributions**.

3. **RAPPOR + DP-Segmented Model**  
   Trained on **structurally anonymized categorical segments** produced by
   label-aware DP segmentation.

All models are trained using **one-hot encoding + Decision Trees**, with
**Logistic Regression used as a secondary sanity-check classifier**.

---

### **Evaluation Dimensions**

Beyond predictive performance, we evaluate multiple dimensions of privacy leakage:

- **Learning Utility**
  - F1 score (positive class)

- **Distribution Distortion**
  - Jensen–Shannon (JS) divergence

- **Structural Leakage**
  - Tree depth  
  - Leaf count  
  - Tree shape similarity (Jaccard)

- **Information Leakage**
  - Mutual Information between feature and label

All learning is performed in a **simulated federated setting** in which the
server never observes raw client categories and all learning operates on
**locally privatized data only**.


# Helper functions for our pipelines (BELOW)

In [10]:
##############################################################
# LABEL-AWARE DP SEGMENTATION & MODELING UTILITIES
#
# This section provides:
#   - Category-level label statistics
#   - Label-aware segmentation scoring
#   - Decision tree + logistic regression training on 1D categorical features
##############################################################

def category_label_counts(X, y, K):
    '''
    Computes per-category counts and label frequencies.

    Parameters:
        X (array-like):
            Category ID per sample.
        y (array-like):
            Binary label (0/1).
        K (int):
            Number of categories.

    Return value(s):
        cat_total (np.array):
            Samples per category.
        cat_pos (np.array):
            Positive labels per category.
    '''

    cat_total = np.zeros(K, dtype=int)
    cat_pos   = np.zeros(K, dtype=int)

    for xi, yi in zip(X, y):
        cat_total[xi] += 1
        if yi == 1:
            cat_pos[xi] += 1

    return cat_total, cat_pos


def label_aware_scores(est_probs, cat_rate, alpha=0.7):
    '''
    Combines:
        - estimated category frequency
        - label prevalence

    Produces a single score used for segmentation ordering.

    Parameters:
        est_probs (array-like):
            RAPPOR-decoded distribution.
        cat_rate (array-like):
            Positive label fraction per category.
        alpha (float):
            Weight assigned to label influence.

    Return value(s):
        scores (np.array):
            Combined category score.
    '''

    est_probs = np.asarray(est_probs, dtype=float)
    cat_rate  = np.asarray(cat_rate, dtype=float)

    # normalize decoded probability vector
    if est_probs.sum() > 0:
        p_norm = est_probs / est_probs.sum()
    else:
        p_norm = np.ones_like(est_probs) / len(est_probs)

    # normalize label prevalence into 0..1 range
    r_min = cat_rate.min()
    r_max = cat_rate.max()
    if r_max > r_min:
        r_norm = (cat_rate - r_min) / (r_max - r_min)
    else:
        r_norm = np.zeros_like(cat_rate)

    # blend rate and frequency
    scores = alpha * r_norm + (1.0 - alpha) * p_norm

    return scores


def dp_segment_label_aware(est_probs, cat_rate, B=30, alpha=0.7):
    '''
    Performs segmentation using both:
        - privacy-preserved frequencies
        - class skew information

    Parameters:
        est_probs (np.array):
            RAPPOR-estimated probabilities.
        cat_rate (np.array):
            Positive-class ratios.
        B (int):
            Number of target segments.
        alpha (float):
            Label weighting strength.

    Return value(s):
        segment_id (np.array):
            Category-to-segment mapping.
        segments (list):
            Grouped category indices.
    '''

    scores = label_aware_scores(est_probs, cat_rate, alpha=alpha)

    # cluster categories by combined utility score
    segment_id, segments = dp_segment_by_values(scores, B=B)

    return segment_id, segments

##############################################################
# HELPER: DOMAIN COMPRESSION FOR RESOLUTION SWEEPS
##############################################################
def compress_categories_by_id(X_cat, K_new, K_global):
    """
    Reduces category cardinality by integer binning.

    mapping[c] = floor(c * K_new / K_global)

    Parameters:
        X_cat (array-like): Original category IDs.
        K_new (int):        Target resolution.
        K_global (int):     Original cardinality.

    Return value(s):
        X_comp (np.array):  Compressed category IDs.
        K_new_eff (int):    Effective number of bins.
    """
    K_new_eff = min(K_new, K_global)
    mapping = (np.arange(K_global) * K_new_eff) // K_global
    return mapping[X_cat], K_new_eff


##############################################################
# SUPERVISED LEARNING ON 1D CATEGORICAL FEATURES
##############################################################

def train_eval_tree_1d(X_cat, y, max_depth=None, random_state=42, test_size=0.3):
    """
    Train a decision tree on 1D categorical IDs using one-hot encoding.
    Returns F1 score (positive class) and the fitted pipeline.

    X_cat: 1D array of category IDs (int)
    y:     1D array of 0/1 labels
    """
    X_cat = np.asarray(X_cat)
    y = np.asarray(y)

    # 1D -> 2D for sklearn
    X = X_cat.reshape(-1, 1)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    # OneHotEncoder + DecisionTree pipeline
    pipe = make_pipeline(
        OneHotEncoder(handle_unknown="ignore"),
        DecisionTreeClassifier(
            max_depth=max_depth,
            class_weight="balanced",  # push tree to care about positives
            random_state=random_state
        )
    )

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    f1 = f1_score(y_test, y_pred, average="binary", pos_label=1)

    return f1, pipe

def train_eval_logreg_1d(X_cat, y, random_state=42, test_size=0.3):
    """
    Train logistic regression on 1D categorical IDs using one-hot encoding.
    This avoids treating the category integers as numeric/ordinal.
    Returns F1 + fitted pipeline.
    """
    X_cat = np.asarray(X_cat)
    y = np.asarray(y)

    X = X_cat.reshape(-1, 1)  # shape (n, 1)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    # OneHotEncoder + LogisticRegression pipeline
    pipe = make_pipeline(
        OneHotEncoder(handle_unknown="ignore"),
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",  # helps with imbalance
            random_state=random_state
        )
    )

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    f1 = f1_score(y_test, y_pred, average="binary", pos_label=1)

    return f1, pipe

In [11]:
##############################################################
# MODEL EXPORT, REPORTING, & STRUCTURAL PRIVACY ANALYSIS
#
# Provides utilities for:
#   - Tree -> JSON conversion
#   - Experiment summary generation
#   - Structural leakage quantification
##############################################################

def sklearn_tree_to_json(model):
    """
    Accepts either:
      - a bare DecisionTreeClassifier, or
      - a Pipeline whose one step is a DecisionTreeClassifier,

    and returns a recursive JSON representation with the form:

        {
          "leaf": True/False,
          "value": [...],
          "feature": int,       # internal only
          "threshold": float,   # internal only
          "left": {...},        # internal only
          "right": {...}        # internal only
        }
    """

    # --- unwrap the decision tree from the model/pipeline ---
    if hasattr(model, "tree_"):
        # bare DecisionTreeClassifier
        decision_tree = model
    elif isinstance(model, Pipeline):
        decision_tree = None
        for _, step in model.steps:
            if hasattr(step, "tree_"):
                decision_tree = step
                break
        if decision_tree is None:
            raise ValueError(
                "Pipeline does not contain a DecisionTree-like step with .tree_."
            )
    else:
        raise TypeError(
            f"Expected a DecisionTreeClassifier or Pipeline, got {type(model)}"
        )

    tree_ = decision_tree.tree_

    # --- recursive conversion to JSON ---
    def recurse(node):
        # leaf node
        if tree_.feature[node] == _tree.TREE_UNDEFINED:
            return {
                "leaf": True,
                "value": tree_.value[node][0].tolist()
            }

        # internal node
        return {
            "leaf": False,
            "feature": int(tree_.feature[node]),
            "threshold": float(tree_.threshold[node]),
            "value": tree_.value[node][0].tolist(),
            "left": recurse(tree_.children_left[node]),
            "right": recurse(tree_.children_right[node])
        }

    return recurse(0)


def write_experiment_readme(drug_dir, meta, metrics):
    '''
    Writes a little summary of experiment results.

    Records:
        - dataset configuration
        - segmentation parameters
        - model performance
        - structural privacy metrics
        - mutual information (inference risk proxy)
    '''

    name = meta.get('drug', meta.get('group_name', ''))
    name_str = name.upper() if isinstance(name, str) else str(name)

    readme = f"""
EXPERIMENT SUMMARY: {name_str}

===========================================
DATASET
-------------------------------------------
Total samples: {meta['n']}
Positive prevalence: {meta['prevalence']:.4f}

===========================================
MODEL CONFIGURATION
-------------------------------------------
Max depth: {meta['max_depth']}
DP segments (B): {meta['B']}
Segment smoothing alpha: {meta['alpha']}

===========================================
MODEL PERFORMANCE (Decision Tree, F1)
-------------------------------------------
Baseline F1:      {metrics['f1_baseline']:.4f}
RAPPOR-only F1:   {metrics['f1_rappor']:.4f}
RAPPOR + DP-seg:  {metrics['f1_dpseg']:.4f}

F1 deltas (tree):
  • RAPPOR - Baseline: {metrics['delta_rappor']:.4f}
  • DPseg  - Baseline: {metrics['delta_dpseg']:.4f}

===========================================
MODEL PERFORMANCE (Logistic Regression, F1)
-------------------------------------------
Baseline logreg F1:      {metrics['f1_logreg_baseline']:.4f}
RAPPOR-only logreg F1:   {metrics['f1_logreg_rappor']:.4f}
RAPPOR + DP-seg logreg:  {metrics['f1_logreg_dpseg']:.4f}

F1 deltas (logreg):
  • RAPPOR - Baseline: {metrics['delta_logreg_rappor']:.4f}
  • DPseg  - Baseline: {metrics['delta_logreg_dpseg']:.4f}

===========================================
STRUCTURAL LEAKAGE REPORT
-------------------------------------------
Baseline leaf count:      {metrics['leaves_base']}
RAPPOR leaf count:        {metrics['leaves_rappor']}
DP-seg leaf count:        {metrics['leaves_dpseg']}

Max depth:
  Baseline: {metrics['depth_base']}
  RAPPOR:   {metrics['depth_rappor']}
  DP-seg:   {metrics['depth_dpseg']}

Tree shape similarity:
  Baseline <-> RAPPOR: {metrics['sim_rappor']:.4f}
  Baseline <-> DP-seg: {metrics['sim_dpseg']:.4f}

===========================================
INFORMATION LEAKAGE (MUTUAL INFORMATION)
-------------------------------------------
Baseline MI: {metrics.get('mi_baseline', float('nan')):.6f}
RAPPOR MI:   {metrics.get('mi_rappor', float('nan')):.6f}
DP-seg MI:   {metrics.get('mi_dpseg', float('nan')):.6f}

MI ratios (relative to baseline):
  RAPPOR / Baseline: {metrics.get('mi_ratio_rappor', float('nan')):.4f}
  DP-seg / Baseline: {metrics.get('mi_ratio_dpseg', float('nan')):.4f}

===========================================
FILES
-------------------------------------------
baseline_tree.json
rappor_tree.json
dpseg_tree.json

baseline_tree_model.joblib
rappor_tree_model.joblib
dpseg_tree_model.joblib

baseline_logreg_model.joblib
rappor_logreg_model.joblib
dpseg_logreg_model.joblib

metadata.json

===========================================
NOTES
-------------------------------------------
Lower similarity scores indicate stronger
structural privacy effects.
Fewer leaves indicate reduced memorization risk.
Lower MI ratios indicate stronger privacy.
    """

    with open(os.path.join(drug_dir, "README.txt"), "w", encoding="utf-8") as f:
        f.write(readme.strip())


def count_leaves(tree_json):
    '''
    Returns number of leaf nodes in tree JSON.
    Smaller trees leak less structure.
    '''
    if tree_json.get("leaf", False):
        return 1
    # defensive: if "leaf" missing, assume internal if left/right present
    return count_leaves(tree_json["left"]) + count_leaves(tree_json["right"])


def tree_depth(tree_json):
    '''
    Returns maximum tree depth.
    '''
    if tree_json.get("leaf", False):
        return 1
    return 1 + max(
        tree_depth(tree_json["left"]),
        tree_depth(tree_json["right"])
    )


def flatten_tree(tree_json):
    '''
    Converts tree structure into a linearized representation
    for comparison. Only structure is preserved, not thresholds.
    '''

    structure = []

    def walk(node):
        if node.get("leaf", False):
            structure.append("L")
            return
        structure.append("N")
        walk(node["left"])
        walk(node["right"])

    walk(tree_json)
    return structure


def jaccard_similarity(a, b):
    '''
    Measures similarity between flattened trees.
    '''
    A = set(enumerate(a))
    B = set(enumerate(b))
    return len(A & B) / max(len(A | B), 1)


def compute_mutual_information(X, y):
    '''
    Computes MI between:
        - feature and label

    Higher MI = higher potential inference risk.
    '''
    return mutual_info_score(X, y)


## **Global RAPPOR Distortion Analysis (Distribution-Level Privacy Cost)**

Before evaluating any supervised learning performance, we first measure the
**distribution-level distortion introduced by RAPPOR alone**, independent of:

- labels  
- segmentation  
- model training  

Using the full client population, we compare:

- the **true categorical distribution**
- the **RAPPOR-decoded distribution**

under a fixed privacy budget ε.

We report the following population-level privacy–utility metrics:

- **Jensen–Shannon (JS) divergence**  
  Measures how much the decoded distribution deviates from the true distribution.

- **Category Match Rate**  
  Fraction of categories that match when resampling from the decoded distribution.

- **Category Hamming Distance**  
  One minus the match rate; a measure of structural category distortion.

These metrics isolate the **privacy cost of RAPPOR itself**, separate from any
learning or segmentation effects.


In [12]:
##############################################################
# GLOBAL RAPPOR DISTORTION BENCHMARK
#
# Measures the population-level distribution distortion
# introduced by RAPPOR under a fixed ε.
#
# IMPORTANT:
#   - Uses the TRUE high-cardinality categories (X_cat)
#   - Does NOT use compressed or DP-segmented features
##############################################################

print("=======================================")
print("      GLOBAL RAPPOR DISTORTION")
print("=======================================\n")


# --------------------------------------------------
# Benchmark privacy budget
# --------------------------------------------------
EPS_BENCH = 3.0
Q_bench   = q_from_epsilon(EPS_BENCH, p=P)
print(f"Benchmark epsilon ε = {EPS_BENCH:.2f}, resulting Q = {Q_bench:.4f}\n")

K_new_target = 128  # or 64, 256 depending on what you want to try (pick ur posion)
X_cat_comp, K_new_eff = compress_categories_by_id(X_cat, K_new_target, K_global)

print("Compressed categories:", K_new_eff)
print("Example counts:\n", pd.Series(X_cat_comp).value_counts().head())

# --------------------------------------------------
# Use true categories
# --------------------------------------------------
X_all    = X_cat_comp      # compressed ids
K_global = K_new_eff       # e.g. 128


# --------------------------------------------------
# True distribution
# --------------------------------------------------
true_counts_global = np.bincount(X_all, minlength=K_global)
true_probs_global  = true_counts_global / true_counts_global.sum()


# --------------------------------------------------
# RAPPOR encode -> aggregate -> decode
# --------------------------------------------------
reports_global = rappor_client_reports(
    X_all,
    k=BLOOM_K,
    h=BLOOM_H,
    f=F,
    p=P,
    q=Q_bench,
    seed=999,
)

counts_global, n_global = rappor_aggregate(reports_global)

est_counts_global, est_probs_global = rappor_decode(
    counts_global,
    n_global,
    K_global,
    BLOOM_K,
    BLOOM_H,
    F,
    P,
    Q_bench,
)

# Safety normalization
est_probs_global = np.clip(est_probs_global, 0, None)
est_probs_global = est_probs_global / est_probs_global.sum()

# --------------------------------------------------
# Distortion Metrics
# --------------------------------------------------
js_global = js_divergence(true_probs_global, est_probs_global)
print(f"JS divergence (global): {js_global:.6f}")

rng = np.random.default_rng(2025)
X_rappor_global = rng.choice(K_global, size=len(X_all), p=est_probs_global)

category_match_global   = np.mean(X_rappor_global == X_all)
category_hamming_global = 1 - category_match_global

# --------------------------------------------------
# Print Results
# --------------------------------------------------
print(f"Category match rate (global): {category_match_global:.4f}")
print(f"Category distance (global):   {category_hamming_global:.4f}")
print("\n(Computed for ε = %.2f; no need to repeat per-drug.)" % EPS_BENCH)


# --------------------------------------------------
# Save benchmark results 
# --------------------------------------------------
GLOBAL_RESULTS_DIR = Path("proj_results/global_rappor_benchmark")
GLOBAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

benchmark_results = {
    "epsilon": float(EPS_BENCH),
    "Q": float(Q_bench),
    "K": int(K_global),
    "JS_divergence": float(js_global),
    "category_match_rate": float(category_match_global),
    "category_hamming_distance": float(category_hamming_global),
}

with open(GLOBAL_RESULTS_DIR / "global_rappor_distortion.json", "w") as f:
    json.dump(benchmark_results, f, indent=2)

print("Global RAPPOR benchmark saved to:")
print(GLOBAL_RESULTS_DIR / "global_rappor_distortion.json")

      GLOBAL RAPPOR DISTORTION

Benchmark epsilon ε = 3.00, resulting Q = 0.9526

Compressed categories: 128
Example counts:
 6    18429
2    11737
1    10077
7     8722
0     8234
dtype: int64
JS divergence (global): 0.093546
Category match rate (global): 0.0301
Category distance (global):   0.9699

(Computed for ε = 3.00; no need to repeat per-drug.)
Global RAPPOR benchmark saved to:
proj_results\global_rappor_benchmark\global_rappor_distortion.json


## **Full ε-Conditioned Privacy–Utility Pipeline Across 14 Drug Outcomes**

This section executes the **complete ε-conditioned end-to-end privacy pipeline**
for all **14 drug-use outcomes**.  
Each outcome is evaluated independently under multiple **local privacy budgets**:

\[
\varepsilon \in \{1.0,\; 3.0,\; 5.0,\; 10.0\}
\]

For **each ε**, the pipeline performs the following stages:

---

### **1. Global RAPPOR Encode–Decode (Per ε)**

Before any model training, a **population-level privatization benchmark** is computed:

- Bloom-filter encoding with RAPPOR
- Server-side aggregation and decoding
- Distribution distortion metrics:
  - Jensen–Shannon (JS) divergence  
  - Category match rate  
  - Hamming distance  

This establishes a privacy baseline **independent of learning**.

---

### **2. Per-Drug Model Training (14 Outcomes × Per ε)**

For each drug outcome and each ε:

- **Baseline (no privacy)**
- **RAPPOR-only**
- **RAPPOR + label-aware DP segmentation**

Each variant is trained using:

- Decision trees
- Logistic regression

---

### **3. Per-Drug Outputs Saved to Disk (Per ε)**

For every drug under each ε, the following artifacts are saved:

- `.joblib` models:
  - baseline, RAPPOR-only, DP-seg
- JSON decision-tree structures
- Tree depth + leaf count
- Jaccard structural similarity vs baseline
- Mutual information leakage (feature ↔ label)
- Category distortion (match + Hamming)
- Auto-generated experiment `README.txt`

---

### **4. Global Drug-Group Models (Per ε)**

In addition to individual drugs, **three global drug groups** are evaluated:

- illicit  
- nicotine  
- prescription_misuse  

Each group is processed with the same three pipelines:
baseline → RAPPOR → RAPPOR + DP-seg.

---

### **5. Resolution Sweeps (Per ε)**

For both:

- Individual drugs  
- Global drug groups  

A full **domain-resolution sweep** is performed:

\[
K_{new} \in \{639\; 500\; 200\; 100\; 50\; 30\}
\]

At each resolution, the following are logged:

- F1 (baseline, RAPPOR, DP-seg)
- JS divergence
- Mutual information
- Privacy–utility composite score

---

### **6. Automatic Best-Model Selection (Per ε)**

For both drugs and groups:

- Best resolution by **privacy–utility score**
- Best resolution by **mutual-information leakage**
- Best summaries exported as CSV

---

### **7. Global File Organization**

All outputs are automatically stored in ther below structure:

<h3>Experiment Output Directory Structure</h3>
<pre>
proj_results/
├── epsilon_1.0/
│   ├── saved_trees/
│   │   ├── marijuana/
│   │   ├── cocaine/
│   │   └── groups/
│   └── resolution_sweep/
│       ├── per_drug/
│       ├── overall/
│       └── groups/
├── epsilon_3.0/
├── epsilon_5.0/
└── epsilon_10.0/
</pre>

This structure enables full **ε-specific privacy–utility optimization**,  
**model-structure auditing**, and **post-hoc visualization**.

---

This design ensures:

- No privacy leakage across ε values  
- Reproducible per-ε experiments  
- Publication-ready experimental organization  

In [13]:
##############################################################
# GLOBAL OUTPUT CONFIGURATION
##############################################################

FINAL_RESULTS_DIR = "proj_results"      
EPS_VALUES = [1.0, 3.0, 5.0, 10.0]
ALL_DRUGS = list(DRUG_VARIABLES.keys())

os.makedirs(FINAL_RESULTS_DIR, exist_ok=True)

##############################################################
# DRUG GROUP DEFINITIONS
##############################################################

DRUG_GROUPS = {
    "illicit": [
        "marijuana", "cocaine", "crack", "heroin",
        "methamphetamine", "inhalants", "hallucinogens"
    ],
    "nicotine": [
        "tobacco", "nicotine_vaping"
    ],
    "prescription_misuse": [
        "opioid_misuse", "tranquilizer_misuse",
        "sedative_misuse", "stimulant_misuse",
        "pain_reliever_misuse"
    ]
}

In [14]:
##############################################################
# MASTER EPSILON SWEEP DRIVER
#
# This is the main experimental controller for the entire
# privacy–utility study.
#
# For each privacy budget ε ∈ {1, 3, 5, 10}, this driver:
#
#   1. Configures RAPPOR parameters from ε
#   2. Computes global population-level RAPPOR distortion
#      (JS divergence, match rate, Hamming distance)
#   3. Saves ε-level metadata for reproducibility
#   4. Trains per-drug models:
#        - Baseline (no privacy)
#        - RAPPOR-only
#        - RAPPOR + label-aware DP segmentation
#   5. Exports for each drug:
#        - .joblib models
#        - JSON tree structures
#        - Tree depth & leaf count
#        - Jaccard structural similarity
#        - Mutual information (leakage proxy)
#        - Auto-generated experiment README
#   6. Trains and evaluates global drug-group models
#   7. Runs full K-resolution sweeps (per drug & per group)
#   8. Saves all summaries and best-K selections to disk
#
# Output structure:
#   proj_results/
#     ├── epsilon_1.0/
#     ├── epsilon_3.0/
#     ├── epsilon_5.0/
#     ├── epsilon_10.0/
#     └── best_epsilon_per_drug.csv
#
# This block fully automates ε-conditioned privacy–utility
# optimization and structural leakage auditing.
##############################################################

BASE_RESULTS_DIR = FINAL_RESULTS_DIR
os.makedirs(BASE_RESULTS_DIR, exist_ok=True)

# Resolutions to evaluate (still doing K sweeps per epsilon)
ALL_K_NEW = [639, 500, 200, 100, 50, 30]

for EPS_TARGET in EPS_VALUES:
    print("\n=======================================================")
    print(f"    RUNNING FULL PIPELINE FOR EPSILON = {EPS_TARGET}")
    print("=======================================================\n")

    # -------------------------------------------
    # 1. Set RAPPOR parameters for this epsilon
    # -------------------------------------------
    BLOOM_K = 128
    BLOOM_H = 2
    F       = 0.5
    P       = 0.5
    Q       = q_from_epsilon(EPS_TARGET, p=P)

    EPSILON = rappor_epsilon(P, Q)
    print(f"Using P={P:.2f}, Q={Q:.4f} -> ε ≈ {EPSILON:.4f}")

    # Local aliases so we always pass params explicitly
    k = BLOOM_K
    h = BLOOM_H
    f_noise = F
    p = P
    q = Q

    # -------------------------------------------
    # 2. Set directories for this epsilon run
    # -------------------------------------------
    EPS_DIR = os.path.join(BASE_RESULTS_DIR, f"epsilon_{EPS_TARGET}")
    os.makedirs(EPS_DIR, exist_ok=True)

    # per-epsilon top-level dirs:
    ROOT_DIR = os.path.join(EPS_DIR, "saved_trees")
    os.makedirs(ROOT_DIR, exist_ok=True)

    SWEEP_ROOT   = os.path.join(EPS_DIR, "resolution_sweep")
    PER_DRUG_DIR = os.path.join(SWEEP_ROOT, "per_drug")
    OVERALL_DIR  = os.path.join(SWEEP_ROOT, "overall")
    os.makedirs(PER_DRUG_DIR, exist_ok=True)
    os.makedirs(OVERALL_DIR, exist_ok=True)

    GROUP_ROOT_DIR    = os.path.join(ROOT_DIR, "groups")
    GROUP_SWEEP_ROOT  = os.path.join(SWEEP_ROOT, "groups")
    GROUP_OVERALL_DIR = os.path.join(GROUP_SWEEP_ROOT, "overall")
    os.makedirs(GROUP_ROOT_DIR, exist_ok=True)
    os.makedirs(GROUP_SWEEP_ROOT, exist_ok=True)
    os.makedirs(GROUP_OVERALL_DIR, exist_ok=True)

    # -------------------------------------------
    # 3a. GLOBAL RAPPOR DISTORTION (per epsilon)
    # -------------------------------------------
    print("=======================================")
    print("      GLOBAL RAPPOR DISTORTION")
    print("=======================================\n")

    X_all    = X_filtered
    K_global = len(mapping_features)

    true_counts_global = np.bincount(X_all, minlength=K_global)
    true_probs_global  = true_counts_global / true_counts_global.sum()

    reports_global = rappor_client_reports(
        X_cat=X_all,
        k=k,
        h=h,
        f=f_noise,
        p=p,
        q=q,
        seed=999
    )
    counts_global, n_global = rappor_aggregate(reports_global)
    est_counts_global, est_probs_global = rappor_decode(
        counts_global,
        n_global,
        K_global,
        k=k,
        h=h,
        f=f_noise,
        p=p,
        q=q
    )

    est_probs_global = np.clip(est_probs_global, 0, None)
    est_probs_global = est_probs_global / est_probs_global.sum()

    js_global = js_divergence(true_probs_global, est_probs_global)
    print(f"[ε={EPS_TARGET}] JS divergence (global): {js_global:.6f}")

    rng = np.random.default_rng(2025)
    X_rappor_global = rng.choice(K_global, size=len(X_all), p=est_probs_global)
    category_match_global   = np.mean(X_rappor_global == X_all)
    category_hamming_global = 1 - category_match_global

    print(f"[ε={EPS_TARGET}] Category match rate (global): {category_match_global:.4f}")
    print(f"[ε={EPS_TARGET}] Category distance (global):   {category_hamming_global:.4f}")
    print("\n(Computed once per epsilon.)")

    print("=======================================")
    print("      GLOBAL ε PER METADATA")
    print("=======================================\n")
    
    epsilon_metadata = {
        "epsilon": float(EPS_TARGET),
        "P": float(P),
        "Q": float(Q),
        "bloom_k": int(BLOOM_K),
        "bloom_h": int(BLOOM_H),
        "f": float(F),
        "global_js": float(js_global),
        "global_category_match": float(category_match_global),
        "global_category_hamming": float(category_hamming_global),
        "num_drugs": len(ALL_DRUGS),
        "num_groups": len(DRUG_GROUPS)
    }

    with open(os.path.join(EPS_DIR, "epsilon_metadata.json"), "w") as f:
        json.dump(epsilon_metadata, f, indent=2)
        
    print("\nGlobal ε per metasata saved!" ) 
    ##############################################################
    # 3b. CLEAN PER-DRUG PIPELINE (TREE + LOGREG, full resolution)
    ##############################################################
    per_drug_model_results = []
    ALL_DRUGS = list(DRUG_VARIABLES.keys())

    for drug in ALL_DRUGS:

        print("\n==========================================================")
        print(f"                 DRUG: {drug.upper()}")
        print("==========================================================\n")

        # create output folder for this drug under this epsilon
        drug_dir = os.path.join(ROOT_DIR, drug.lower())
        os.makedirs(drug_dir, exist_ok=True)

        # 1. labels + features
        y_raw, mask_y = make_drug_labels(df_filtered, drug)
        X_cat = X_filtered[mask_y]
        y     = y_raw

        n = len(y)
        prevalence = float(y.mean())

        print(f"n = {n}")
        print(f"Prevalence = {prevalence:.4f}\n")

        # 2. Baseline models (tree + logreg)
        f1_tree_baseline, tree_base = train_eval_tree_1d(
            X_cat, y, max_depth=None, random_state=42
        )
        f1_logreg_baseline, logreg_base = train_eval_logreg_1d(
            X_cat, y, random_state=42
        )

        print(f"Baseline tree F1:           {f1_tree_baseline:.4f}")
        print(f"Baseline logreg F1:         {f1_logreg_baseline:.4f}")

        # 3. RAPPOR-only models (global decoded distribution)
        rng = np.random.default_rng(hash(drug) % 10_000)
        X_rappor_user = rng.choice(K_global, size=n, p=est_probs_global)

        f1_tree_rappor, tree_rappor = train_eval_tree_1d(
            X_rappor_user, y, max_depth=None, random_state=42
        )
        f1_logreg_rappor, logreg_rappor = train_eval_logreg_1d(
            X_rappor_user, y, random_state=42
        )

        print(f"RAPPOR-only tree F1:        {f1_tree_rappor:.4f}")
        print(f"RAPPOR-only logreg F1:      {f1_logreg_rappor:.4f}")

        # 4. DP-seg models (on top of RAPPOR)
        cat_total, cat_pos = category_label_counts(X_cat, y, K_global)
        cat_rate = np.zeros(K_global)
        mask_nonzero = (cat_total > 0)
        cat_rate[mask_nonzero] = cat_pos[mask_nonzero] / cat_total[mask_nonzero]

        B = 30
        alpha = 0.7
        segment_id, segments = dp_segment_label_aware(
            est_probs_global, cat_rate, B=B, alpha=alpha
        )

        X_seg = segment_id[X_rappor_user]

        f1_tree_dpseg, tree_dpseg = train_eval_tree_1d(
            X_seg, y, max_depth=None, random_state=42
        )
        f1_logreg_dpseg, logreg_dpseg = train_eval_logreg_1d(
            X_seg, y, random_state=42
        )

        print(f"RAPPOR + DP-seg tree F1:    {f1_tree_dpseg:.4f}")
        print(f"RAPPOR + DP-seg logreg F1:  {f1_logreg_dpseg:.4f}\n")

        # 5. Save models
        dump(tree_base,   os.path.join(drug_dir, "baseline_tree_model.joblib"))
        dump(tree_rappor, os.path.join(drug_dir, "rappor_tree_model.joblib"))
        dump(tree_dpseg,  os.path.join(drug_dir, "dpseg_tree_model.joblib"))

        dump(logreg_base,   os.path.join(drug_dir, "baseline_logreg_model.joblib"))
        dump(logreg_rappor, os.path.join(drug_dir, "rappor_logreg_model.joblib"))
        dump(logreg_dpseg,  os.path.join(drug_dir, "dpseg_logreg_model.joblib"))

        # 6. JSON trees + metadata
        baseline_json = sklearn_tree_to_json(tree_base)
        rappor_json   = sklearn_tree_to_json(tree_rappor)
        dpseg_json    = sklearn_tree_to_json(tree_dpseg)

        with open(os.path.join(drug_dir, "baseline_tree.json"), "w") as f:
            json.dump(baseline_json, f, indent=4)
        with open(os.path.join(drug_dir, "rappor_tree.json"), "w") as f:
            json.dump(rappor_json, f, indent=4)
        with open(os.path.join(drug_dir, "dpseg_tree.json"), "w") as f:
            json.dump(dpseg_json, f, indent=4)

        meta_info = {
            "drug": drug,
            "n": n,
            "prevalence": prevalence,
            "B": B,
            "alpha": alpha,
            "max_depth": None,
            "epsilon": float(EPS_TARGET),
            "js_global": float(js_global)
        }

        leaves_base   = count_leaves(baseline_json)
        leaves_rappor = count_leaves(rappor_json)
        leaves_dpseg  = count_leaves(dpseg_json)

        depth_base   = tree_depth(baseline_json)
        depth_rappor = tree_depth(rappor_json)
        depth_dpseg  = tree_depth(dpseg_json)

        struct_base   = flatten_tree(baseline_json)
        struct_rappor = flatten_tree(rappor_json)
        struct_dpseg  = flatten_tree(dpseg_json)

        sim_rappor = jaccard_similarity(struct_base, struct_rappor)
        sim_dpseg  = jaccard_similarity(struct_base, struct_dpseg)

        meta_info["sim_baseline_rappor"] = float(sim_rappor)
        meta_info["sim_baseline_dpseg"]  = float(sim_dpseg)

        with open(os.path.join(drug_dir, "metadata.json"), "w") as f:
            json.dump(meta_info, f, indent=4)

        # 7. Distortion + MI
        category_match_rappor   = float(np.mean(X_rappor_user == X_cat))
        category_hamming_rappor = 1 - category_match_rappor

        seg_true = segment_id[X_cat]
        category_match_dpseg   = float(np.mean(X_seg == seg_true))
        category_hamming_dpseg = 1 - category_match_dpseg

        mi_baseline = compute_mutual_information(X_cat, y)
        mi_rappor   = compute_mutual_information(X_rappor_user, y)
        mi_dpseg    = compute_mutual_information(X_seg, y)

        mi_ratio_rappor = mi_rappor / mi_baseline if mi_baseline > 0 else 0.0
        mi_ratio_dpseg  = mi_dpseg  / mi_baseline if mi_baseline > 0 else 0.0

        metrics = {
            "f1_baseline":          f1_tree_baseline,
            "f1_rappor":            f1_tree_rappor,
            "f1_dpseg":             f1_tree_dpseg,
            "f1_logreg_baseline":   f1_logreg_baseline,
            "f1_logreg_rappor":     f1_logreg_rappor,
            "f1_logreg_dpseg":      f1_logreg_dpseg,
            "delta_rappor":          f1_tree_rappor - f1_tree_baseline,
            "delta_dpseg":           f1_tree_dpseg - f1_tree_baseline,
            "delta_logreg_rappor":   f1_logreg_rappor - f1_logreg_baseline,
            "delta_logreg_dpseg":    f1_logreg_dpseg - f1_logreg_baseline,
            "leaves_base":           leaves_base,
            "leaves_rappor":         leaves_rappor,
            "leaves_dpseg":          leaves_dpseg,
            "depth_base":            depth_base,
            "depth_rappor":          depth_rappor,
            "depth_dpseg":           depth_dpseg,
            "sim_rappor":            sim_rappor,
            "sim_dpseg":             sim_dpseg,
            "mi_baseline":           mi_baseline,
            "mi_rappor":             mi_rappor,
            "mi_dpseg":              mi_dpseg,
            "mi_ratio_rappor":       mi_ratio_rappor,
            "mi_ratio_dpseg":        mi_ratio_dpseg,
            "match_rappor":          category_match_rappor,
            "hamming_rappor":        category_hamming_rappor,
            "match_dpseg":           category_match_dpseg,
            "hamming_dpseg":         category_hamming_dpseg
        }

        write_experiment_readme(drug_dir, meta_info, metrics)

        # sample mapping print
        idx_sample = np.random.choice(n, size=min(15, n), replace=False)
        print("Sample mapping (Raw --> RAPPOR --> DP-seg)")
        print("Raw:     ", X_cat[idx_sample])
        print("RAPPOR:  ", X_rappor_user[idx_sample])
        print("DP-seg:  ", X_seg[idx_sample])

        per_drug_model_results.append({
            "drug": drug,
            "n": n,
            "prevalence": prevalence,
            "B": B,
            "alpha": alpha,
            "tree_baseline_f1":      f1_tree_baseline,
            "tree_rappor_f1":        f1_tree_rappor,
            "tree_dpseg_f1":         f1_tree_dpseg,
            "logreg_baseline_f1":    f1_logreg_baseline,
            "logreg_rappor_f1":      f1_logreg_rappor,
            "logreg_dpseg_f1":       f1_logreg_dpseg,
            "js_global":             js_global,
            "epsilon":               EPS_TARGET
        })

    pd.DataFrame(per_drug_model_results).to_csv(
        os.path.join(ROOT_DIR, "per_drug_models_summary.csv"),
        index=False
    )

    ##############################################################
    # 3c. GLOBAL DRUG-GROUP MODELS (TREE + LOGREG, full resolution)
    ##############################################################
    group_model_results = []

    for group_name, members in DRUG_GROUPS.items():

        print("\n==========================================================")
        print(f"            DRUG GROUP: {group_name.upper()}")
        print("==========================================================\n")

        # output folder for this group (under this epsilon)
        group_dir = os.path.join(GROUP_ROOT_DIR, f"group_{group_name}")
        os.makedirs(group_dir, exist_ok=True)

        # 1. GROUP LABELS & FEATURE FILTERING
        y_group, mask_group = make_group_labels(df_filtered, group_name, members)
        X_cat = X_filtered[mask_group]
        y     = y_group

        n = len(y)
        prevalence = float(y.mean())

        print(f"n = {n}")
        print(f"Prevalence = {prevalence:.4f}\n")

        # 2. BASELINE MODELS (RAW CATEGORIES)
        f1_tree_baseline, tree_base = train_eval_tree_1d(
            X_cat, y, max_depth=None, random_state=42
        )
        f1_logreg_baseline, logreg_base = train_eval_logreg_1d(
            X_cat, y, random_state=42
        )

        print(f"Baseline tree F1:           {f1_tree_baseline:.4f}")
        print(f"Baseline logreg F1:         {f1_logreg_baseline:.4f}")

        # 3. RAPPOR-ONLY MODELS (GLOBAL DECODED DISTRIBUTION)
        rng = np.random.default_rng(hash(group_name) % 10_000)
        X_rappor_user = rng.choice(K_global, size=n, p=est_probs_global)

        f1_tree_rappor, tree_rappor = train_eval_tree_1d(
            X_rappor_user, y, max_depth=None, random_state=42
        )
        f1_logreg_rappor, logreg_rappor = train_eval_logreg_1d(
            X_rappor_user, y, random_state=42
        )

        print(f"RAPPOR-only tree F1:        {f1_tree_rappor:.4f}")
        print(f"RAPPOR-only logreg F1:      {f1_logreg_rappor:.4f}")

        # 4. LABEL-AWARE DP SEGMENTATION + DP-SEG MODELS
        cat_total, cat_pos = category_label_counts(X_cat, y, K_global)
        cat_rate = np.zeros(K_global)
        mask_nonzero = (cat_total > 0)
        cat_rate[mask_nonzero] = (
            cat_pos[mask_nonzero] / cat_total[mask_nonzero]
        )

        B = 30
        alpha = 0.7

        segment_id, segments = dp_segment_label_aware(
            est_probs_global, cat_rate, B=B, alpha=alpha
        )

        X_seg = segment_id[X_rappor_user]

        f1_tree_dpseg, tree_dpseg = train_eval_tree_1d(
            X_seg, y, max_depth=None, random_state=42
        )
        f1_logreg_dpseg, logreg_dpseg = train_eval_logreg_1d(
            X_seg, y, random_state=42
        )

        print(f"RAPPOR + DP-seg tree F1:    {f1_tree_dpseg:.4f}")
        print(f"RAPPOR + DP-seg logreg F1:  {f1_logreg_dpseg:.4f}\n")

        # 5. SAVE MODELS (TREE + LOGREG)
        dump(tree_base,   os.path.join(group_dir, "baseline_tree_model.joblib"))
        dump(tree_rappor, os.path.join(group_dir, "rappor_tree_model.joblib"))
        dump(tree_dpseg,  os.path.join(group_dir, "dpseg_tree_model.joblib"))

        dump(logreg_base,   os.path.join(group_dir, "baseline_logreg_model.joblib"))
        dump(logreg_rappor, os.path.join(group_dir, "rappor_logreg_model.joblib"))
        dump(logreg_dpseg,  os.path.join(group_dir, "dpseg_logreg_model.joblib"))

        print(f"[SAVED] Group models (tree + logreg) -> {group_dir}")

        # 6. EXPORT TREE STRUCTURE (JSON)
        baseline_json = sklearn_tree_to_json(tree_base)
        rappor_json   = sklearn_tree_to_json(tree_rappor)
        dpseg_json    = sklearn_tree_to_json(tree_dpseg)

        with open(os.path.join(group_dir, "baseline_tree.json"), "w") as f:
            json.dump(baseline_json, f, indent=4)
        with open(os.path.join(group_dir, "rappor_tree.json"), "w") as f:
            json.dump(rappor_json, f, indent=4)
        with open(os.path.join(group_dir, "dpseg_tree.json"), "w") as f:
            json.dump(dpseg_json, f, indent=4)

        meta_info = {
            "group_name": group_name,
            "members": members,
            "n": n,
            "prevalence": prevalence,
            "B": B,
            "alpha": alpha,
            "max_depth": None,
            "epsilon": float(EPS_TARGET),
            "js_global": float(js_global)
        }

        # 7. STRUCTURAL LEAKAGE + MI
        leaves_base   = count_leaves(baseline_json)
        leaves_rappor = count_leaves(rappor_json)
        leaves_dpseg  = count_leaves(dpseg_json)

        depth_base   = tree_depth(baseline_json)
        depth_rappor = tree_depth(rappor_json)
        depth_dpseg  = tree_depth(dpseg_json)

        struct_base   = flatten_tree(baseline_json)
        struct_rappor = flatten_tree(rappor_json)
        struct_dpseg  = flatten_tree(dpseg_json)

        sim_rappor = jaccard_similarity(struct_base, struct_rappor)
        sim_dpseg  = jaccard_similarity(struct_base, struct_dpseg)

        mi_baseline = compute_mutual_information(X_cat, y)
        mi_rappor   = compute_mutual_information(X_rappor_user, y)
        mi_dpseg    = compute_mutual_information(X_seg, y)

        mi_ratio_rappor = mi_rappor / mi_baseline if mi_baseline > 0 else 0.0
        mi_ratio_dpseg  = mi_dpseg  / mi_baseline if mi_baseline > 0 else 0.0

        meta_info["sim_baseline_rappor"] = float(sim_rappor)
        meta_info["sim_baseline_dpseg"]  = float(sim_dpseg)

        with open(os.path.join(group_dir, "metadata.json"), "w") as f:
            json.dump(meta_info, f, indent=4)

        metrics = {
            "f1_baseline":        f1_tree_baseline,
            "f1_rappor":          f1_tree_rappor,
            "f1_dpseg":           f1_tree_dpseg,
            "f1_logreg_baseline": f1_logreg_baseline,
            "f1_logreg_rappor":   f1_logreg_rappor,
            "f1_logreg_dpseg":    f1_logreg_dpseg,
            "delta_rappor":        f1_tree_rappor - f1_tree_baseline,
            "delta_dpseg":         f1_tree_dpseg - f1_tree_baseline,
            "delta_logreg_rappor": f1_logreg_rappor - f1_logreg_baseline,
            "delta_logreg_dpseg":  f1_logreg_dpseg - f1_logreg_baseline,
            "leaves_base":         leaves_base,
            "leaves_rappor":       leaves_rappor,
            "leaves_dpseg":        leaves_dpseg,
            "depth_base":          depth_base,
            "depth_rappor":        depth_rappor,
            "depth_dpseg":         depth_dpseg,
            "sim_rappor":          sim_rappor,
            "sim_dpseg":           sim_dpseg,
            "mi_baseline":         mi_baseline,
            "mi_rappor":           mi_rappor,
            "mi_dpseg":            mi_dpseg,
            "mi_ratio_rappor":     mi_ratio_rappor,
            "mi_ratio_dpseg":      mi_ratio_dpseg
        }

        write_experiment_readme(group_dir, meta_info, metrics)

        category_match_rappor   = float(np.mean(X_rappor_user == X_cat))
        category_hamming_rappor = 1 - category_match_rappor

        seg_true = segment_id[X_cat]
        category_match_dpseg   = float(np.mean(X_seg == seg_true))
        category_hamming_dpseg = 1 - category_match_dpseg

        group_model_results.append({
            "group": group_name,
            "members": ",".join(members),
            "n": n,
            "prevalence": prevalence,
            "B": B,
            "alpha": alpha,
            "f1_tree_baseline":   f1_tree_baseline,
            "f1_tree_rappor":     f1_tree_rappor,
            "f1_tree_dpseg":      f1_tree_dpseg,
            "f1_logreg_baseline": f1_logreg_baseline,
            "f1_logreg_rappor":   f1_logreg_rappor,
            "f1_logreg_dpseg":    f1_logreg_dpseg,
            "delta_tree_rappor":   f1_tree_rappor - f1_tree_baseline,
            "delta_tree_dpseg":    f1_tree_dpseg - f1_tree_baseline,
            "delta_logreg_rappor": f1_logreg_rappor - f1_logreg_baseline,
            "delta_logreg_dpseg":  f1_logreg_dpseg - f1_logreg_baseline,
            "match_rappor":        category_match_rappor,
            "hamming_rappor":      category_hamming_rappor,
            "match_dpseg":         category_match_dpseg,
            "hamming_dpseg":       category_hamming_dpseg,
            "sim_rappor":          sim_rappor,
            "sim_dpseg":           sim_dpseg,
            "leaves_base":         leaves_base,
            "leaves_rappor":       leaves_rappor,
            "leaves_dpseg":        leaves_dpseg,
            "depth_base":          depth_base,
            "depth_rappor":        depth_rappor,
            "depth_dpseg":         depth_dpseg,
            "mi_baseline":         mi_baseline,
            "mi_rappor":           mi_rappor,
            "mi_dpseg":            mi_dpseg,
            "mi_ratio_rappor":     mi_ratio_rappor,
            "mi_ratio_dpseg":      mi_ratio_dpseg,
            "epsilon":             EPS_TARGET,
            "js_global":           js_global
        })

    pd.DataFrame(group_model_results).to_csv(
        os.path.join(GROUP_ROOT_DIR, "global_group_results.csv"),
        index=False
    )

    ##############################################################
    # 3d. RESOLUTION SWEEP ACROSS ALL DRUGS (per epsilon)
    ##############################################################
    results_all_drugs = []

    print("=======================================================")
    print("     RUNNING FINAL RESOLUTION SWEEP FOR ALL DRUGS")
    print("=======================================================\n")

    for drug in DRUG_VARIABLES.keys():

        print(f"\n=== RESOLUTION SWEEP FOR DRUG: {drug.upper()} ===")

        y_raw, mask_y = make_drug_labels(df_filtered, drug)
        X_cat = X_filtered[mask_y]
        y     = y_raw

        n = len(y)
        prevalence = float(y.mean())
        print(f"n = {n}, prevalence = {prevalence:.4f}")

        per_drug_rows = []
        drug_sweep_dir = os.path.join(PER_DRUG_DIR, drug.lower())
        os.makedirs(drug_sweep_dir, exist_ok=True)

        for K_new in ALL_K_NEW:

            # 1) Compress domain
            X_comp, K_eff = compress_categories_by_id(X_cat, K_new, K_global)

            # 2) Baseline
            f1_baseline, _ = train_eval_tree_1d(
                X_comp, y, max_depth=None, random_state=42
            )

            # 3) True distribution
            true_counts = np.bincount(X_comp, minlength=K_eff)
            true_probs  = true_counts / true_counts.sum()

            # 4) RAPPOR encode/decode
            reports = rappor_client_reports(
                X_cat=X_comp,
                k=k,
                h=h,
                f=f_noise,
                p=p,
                q=q,
                seed=123
            )
            counts, n_rep = rappor_aggregate(reports)
            est_counts, est_probs = rappor_decode(
                counts,
                n_rep,
                K_eff,
                k=k,
                h=h,
                f=f_noise,
                p=p,
                q=q
            )

            est_probs = np.clip(est_probs, 0, None)
            est_probs = est_probs / est_probs.sum() if est_probs.sum() > 0 else np.ones(K_eff) / K_eff

            # 5) Distribution distortion
            js = js_divergence(true_probs, est_probs)

            # 6) RAPPOR-only learning
            rng = np.random.default_rng(42 + K_eff)
            X_rappor_cat = rng.choice(K_eff, size=n, p=est_probs)

            f1_rappor, _ = train_eval_tree_1d(
                X_rappor_cat, y, max_depth=None, random_state=42
            )

            # 7) Label-aware DP segmentation
            cat_total, cat_pos = category_label_counts(X_comp, y, K_eff)
            cat_rate = np.zeros(K_eff)
            mask_nonzero = (cat_total > 0)
            cat_rate[mask_nonzero] = cat_pos[mask_nonzero] / cat_total[mask_nonzero]

            B = min(30, K_eff)
            alpha = 0.7

            segment_id, segments = dp_segment_label_aware(
                est_probs, cat_rate, B=B, alpha=alpha
            )

            X_seg = segment_id[X_rappor_cat]

            # 8) DP-seg tree
            f1_dpseg, _ = train_eval_tree_1d(
                X_seg, y, max_depth=None, random_state=42
            )

            # 9) Mutual information
            mi_baseline = compute_mutual_information(X_comp, y)
            mi_rappor   = compute_mutual_information(X_rappor_cat, y)
            mi_dpseg    = compute_mutual_information(X_seg, y)

            mi_ratio_rappor = mi_rappor / mi_baseline if mi_baseline > 0 else 0.0
            mi_ratio_dpseg  = mi_dpseg  / mi_baseline if mi_baseline > 0 else 0.0

            # 10) Composite score
            privacy_utility_score = f1_dpseg / (1 + js)

            # 11) Log results
            row = {
                "drug": drug,
                "K_new": K_eff,
                "B": B,
                "alpha": alpha,
                "f1_baseline": f1_baseline,
                "f1_rappor": f1_rappor,
                "f1_dpseg": f1_dpseg,
                "mi_baseline": mi_baseline,
                "mi_rappor": mi_rappor,
                "mi_dpseg": mi_dpseg,
                "mi_ratio_rappor": mi_ratio_rappor,
                "mi_ratio_dpseg": mi_ratio_dpseg,
                "privacy_utility_score": privacy_utility_score,
                "js_divergence": js,
                "n": n,
                "prevalence": prevalence,
                "epsilon": EPS_TARGET,
                "js_global": js_global
            }

            results_all_drugs.append(row)
            per_drug_rows.append(row)

            pd.DataFrame(per_drug_rows).to_csv(
                os.path.join(drug_sweep_dir, "resolution_results.csv"),
                index=False
            )

            print(f"[SAVED] Drug {drug} @ K={K_eff} -> {drug_sweep_dir}")

    df_all = pd.DataFrame(results_all_drugs)
    df_all.to_csv(os.path.join(OVERALL_DIR, "all_drugs_resolution.csv"), index=False)

    best_overall = df_all.loc[df_all["privacy_utility_score"].idxmax()]
    print("\n========== BEST K OVERALL (DRUGS) ==========")
    print(best_overall)

    best_per_drug = df_all.loc[df_all.groupby("drug")["privacy_utility_score"].idxmax()]
    best_per_drug.to_csv(os.path.join(OVERALL_DIR, "best_k_per_drug_summary.csv"), index=False)

    best_k_by_mi = df_all.loc[df_all.groupby("drug")["mi_dpseg"].idxmax()]
    best_k_by_mi.to_csv(os.path.join(OVERALL_DIR, "best_k_per_drug_by_mi.csv"), index=False)

    ##############################################################
    # 3e. RESOLUTION SWEEP FOR GLOBAL DRUG GROUPS (per epsilon)
    ##############################################################
    results_all_groups = []

    print("\n=======================================================")
    print("   RUNNING RESOLUTION SWEEP FOR GLOBAL DRUG GROUPS")
    print("=======================================================\n")

    for group_name, members in DRUG_GROUPS.items():

        print(f"\n=== RESOLUTION SWEEP FOR GROUP: {group_name.upper()} ===")
        print(f"Members: {', '.join(members)}")

        y_group, mask_group = make_group_labels(df_filtered, group_name, members)
        X_cat = X_filtered[mask_group]
        y     = y_group

        n = len(y)
        prevalence = float(y.mean())
        print(f"n = {n}, prevalence = {prevalence:.4f}")

        per_group_rows = []
        group_sweep_dir = os.path.join(GROUP_SWEEP_ROOT, f"group_{group_name}")
        os.makedirs(group_sweep_dir, exist_ok=True)

        for K_new in ALL_K_NEW:

            # 1) Compress domain
            X_comp, K_eff = compress_categories_by_id(X_cat, K_new, K_global)

            # 2) Baseline
            f1_baseline, _ = train_eval_tree_1d(
                X_comp, y, max_depth=None, random_state=42
            )

            # 3) True distribution
            true_counts = np.bincount(X_comp, minlength=K_eff)
            true_probs  = true_counts / true_counts.sum()

            # 4) RAPPOR encode/decode
            reports = rappor_client_reports(
                X_cat=X_comp,
                k=k,
                h=h,
                f=f_noise,
                p=p,
                q=q,
                seed=123
            )
            counts, n_rep = rappor_aggregate(reports)
            est_counts, est_probs = rappor_decode(
                counts,
                n_rep,
                K_eff,
                k=k,
                h=h,
                f=f_noise,
                p=p,
                q=q
            )

            est_probs = np.clip(est_probs, 0, None)
            est_probs = est_probs / est_probs.sum() if est_probs.sum() > 0 else np.ones(K_eff) / K_eff

            # 5) Distribution distortion
            js = js_divergence(true_probs, est_probs)

            # 6) RAPPOR-only learning
            rng = np.random.default_rng(42 + K_eff)
            X_rappor_cat = rng.choice(K_eff, size=n, p=est_probs)

            f1_rappor, _ = train_eval_tree_1d(
                X_rappor_cat, y, max_depth=None, random_state=42
            )

            # 7) Label-aware DP segmentation
            cat_total, cat_pos = category_label_counts(X_comp, y, K_eff)
            cat_rate = np.zeros(K_eff)
            mask_nonzero = (cat_total > 0)
            cat_rate[mask_nonzero] = cat_pos[mask_nonzero] / cat_total[mask_nonzero]

            B = min(30, K_eff)
            alpha = 0.7

            segment_id, segments = dp_segment_label_aware(
                est_probs, cat_rate, B=B, alpha=alpha
            )

            X_seg = segment_id[X_rappor_cat]

            # 8) DP-seg tree
            f1_dpseg, _ = train_eval_tree_1d(
                X_seg, y, max_depth=None, random_state=42
            )

            # 9) Mutual information
            mi_baseline = compute_mutual_information(X_comp, y)
            mi_rappor   = compute_mutual_information(X_rappor_cat, y)
            mi_dpseg    = compute_mutual_information(X_seg, y)

            mi_ratio_rappor = mi_rappor / mi_baseline if mi_baseline > 0 else 0.0
            mi_ratio_dpseg  = mi_dpseg  / mi_baseline if mi_baseline > 0 else 0.0

            # 10) Composite privacy–utility score
            privacy_utility_score = f1_dpseg / (1 + js)

            # 11) Log results
            row = {
                "group": group_name,
                "members": ",".join(members),
                "K_new": K_eff,
                "B": B,
                "alpha": alpha,
                "f1_baseline": f1_baseline,
                "f1_rappor": f1_rappor,
                "f1_dpseg": f1_dpseg,
                "mi_baseline": mi_baseline,
                "mi_rappor": mi_rappor,
                "mi_dpseg": mi_dpseg,
                "mi_ratio_rappor": mi_ratio_rappor,
                "mi_ratio_dpseg": mi_ratio_dpseg,
                "privacy_utility_score": privacy_utility_score,
                "js_divergence": js,
                "n": n,
                "prevalence": prevalence,
                "epsilon": EPS_TARGET,
                "js_global": js_global
            }

            results_all_groups.append(row)
            per_group_rows.append(row)

            pd.DataFrame(per_group_rows).to_csv(
                os.path.join(group_sweep_dir, "resolution_results.csv"),
                index=False
            )

            print(f"[SAVED] Group {group_name} @ K={K_eff} -> {group_sweep_dir}")

    df_groups = pd.DataFrame(results_all_groups)
    df_groups.to_csv(
        os.path.join(GROUP_OVERALL_DIR, "all_groups_resolution.csv"),
        index=False
    )

    best_overall_group = df_groups.loc[df_groups["privacy_utility_score"].idxmax()]
    print("\n========== BEST K OVERALL (GROUPS) ==========")
    print(best_overall_group)

    best_per_group = df_groups.loc[
        df_groups.groupby("group")["privacy_utility_score"].idxmax()
    ]
    best_per_group.to_csv(
        os.path.join(GROUP_OVERALL_DIR, "best_k_per_group_summary.csv"),
        index=False
    )

    best_k_by_mi_groups = df_groups.loc[
        df_groups.groupby("group")["mi_dpseg"].idxmax()
    ]
    best_k_by_mi_groups.to_csv(
        os.path.join(GROUP_OVERALL_DIR, "best_k_per_group_by_mi.csv"),
        index=False
    )

    print(f"\n######## FINISHED EPSILON = {EPS_TARGET} ########\n")



    RUNNING FULL PIPELINE FOR EPSILON = 1.0

Using P=0.50, Q=0.7311 -> ε ≈ 1.0000
      GLOBAL RAPPOR DISTORTION

[ε=1.0] JS divergence (global): 0.440903
[ε=1.0] Category match rate (global): 0.0029
[ε=1.0] Category distance (global):   0.9971

(Computed once per epsilon.)
      GLOBAL ε PER METADATA


Global ε per metasata saved!

                 DRUG: MARIJUANA

n = 173729
Prevalence = 0.2363

Baseline tree F1:           0.4583
Baseline logreg F1:         0.4583
RAPPOR-only tree F1:        0.3253
RAPPOR-only logreg F1:      0.3253
RAPPOR + DP-seg tree F1:    0.2882
RAPPOR + DP-seg logreg F1:  0.2882

Sample mapping (Raw --> RAPPOR --> DP-seg)
Raw:      [176 167  60  15  23  15 169  55  30 115 178 278  34  45  47]
RAPPOR:   [ 90 254   7  90 333 484 132 314 185 420 259 317 167 481 104]
DP-seg:   [15 20  7 15 11 28  4  5  2 12 24 15 14 20 18]

                 DRUG: COCAINE

n = 173729
Prevalence = 0.0194

Baseline tree F1:           0.0689
Baseline logreg F1:         0.0690
RAPPOR-o

## **Visualization & Privacy–Utility Tradeoff Analysis**

This section generates the **complete, publication-quality visualization suite** for the ε-conditioned privacy–utility experiment pipeline. All figures are produced directly from the automatically generated resolution-sweep results stored at:

<pre>
proj_results/
├── epsilon_1.0/resolution_sweep/overall/all_drugs_resolution.csv
├── epsilon_3.0/resolution_sweep/overall/all_drugs_resolution.csv
├── epsilon_5.0/resolution_sweep/overall/all_drugs_resolution.csv
└── epsilon_10.0/resolution_sweep/overall/all_drugs_resolution.csv
</pre>

Each visualization quantitatively characterizes **the tradeoff between privacy distortion, predictive utility, and model leakage** across:
- Multiple **privacy budgets** (ε ∈ {1, 3, 5, 10})
- Multiple **category resolutions** (K)
- All **14 drug-use outcomes**

---

## **Structured Output Organization**

All figures are automatically saved under the centralized results directory:

<pre>
proj_results/
└── saved_figures/
    ├── master_summary/
    ├── per_drug_frontiers/
    ├── resolution_sweeps/
    ├── mutual_information/
    ├── structural_similarity/
    ├── tree_visualizations/
    ├── global_distort_vs_k/
    ├── priv_util_vs_K_eps/
    ├── f1_recovery/
    ├── mi_vs_K_eps/
    ├── js_vs_K_eps/
    └── overall_pri_util_vs_eps/
</pre>

This guarantees **clean separation of result types**, making the experimental artifacts easy to audit, visualize, and include in reports.

---

## **Figure Families Generated**

### 1. **Privacy–Utility Score vs Resolution (Per ε)**
Saved to:
> saved_figures/priv_util_vs_K_eps/

Shows how the **combined privacy–utility objective varies as categorical resolution K changes** for each ε.

---

### 2. **F1 Recovery Across Pipelines (Per ε)**
Saved to:
> saved_figures/f1_recovery/

Compares:
- Baseline (no privacy)
- RAPPOR-only
- RAPPOR + DP-segmentation  

This quantifies **utility degradation and recovery under privacy constraints**.

---

### 3. **Mutual Information Leakage vs Resolution (Per ε)**
Saved to:
> saved_figures/mi_vs_K_eps/

Measures **inference leakage** between features and labels as compression and privatization increase.

---

### 4. **JS Divergence vs Resolution (Per ε)**
Saved to:
> saved_figures/js_vs_K_eps/

Captures the **population-level distribution distortion** introduced by RAPPOR under each ε.

---

### 5. **Overall Average Privacy–Utility vs ε**
Saved to:
> saved_figures/overall_pri_util_vs_eps/

Summarizes how increasing privacy budgets globally impacts utility.

---

### 6. **Master Privacy–Utility Tradeoff (JS vs F1)**
Saved to:
> saved_figures/master_summary/
> saved_figures/per_drug_frontiers/

Each point corresponds to a (K, ε) operating point. These plots form the **core experimental evidence of privacy–utility behavior**.

---

### 7. **Global Mutual Information vs Resolution**
Saved to:
> saved_figures/resolution_sweeps/

Shows how **mean information leakage collapses under resolution compression**.

---

### 8. **Global F1 Recovery vs Resolution**
Saved to:
> saved_figures/mutual_information/

Tracks **average predictive performance recovery** across all drugs.

---

### 9. **Global Distribution Distortion vs Resolution**
Saved to:
> saved_figures/global_distort_vs_k/

Quantifies **how far RAPPOR shifts the population distribution** as K shrinks.

---

### 10. **Tree Structural Leakage (Heatmaps)**
Saved to:
> saved_figures/structural_similarity/

Pairwise **Jaccard-based similarity heatmaps** between:
- Baseline
- RAPPOR
- DP-seg trees  

These directly measure **structural leakage**.

---

### 11. **Side-by-Side Decision Tree Visualizations**
Saved to:
> saved_figures/tree_visualizations/

For each drug and ε, includes:
- Baseline tree
- RAPPOR tree
- DP-seg tree  

All models are extracted safely from pipelines to ensure correct rendering.

---

## **Purpose of the Visualization Suite**

Together, these figures help to show:

- **Statistical validity** (F1 recovery)
- **Privacy guarantees** (JS distortion + ε)
- **Information leakage auditing** (mutual information)
- **Structural leakage analysis** (tree similarity)
- **Interpretability** (full decision tree visualizations)
- **Optimization guidance** (privacy–utility)

In [41]:
# ------------------------------------------------------------
# Create output folder for saved figures
# ------------------------------------------------------------

FIGURE_DIR = os.path.join("proj_results", "saved_figures")
os.makedirs(FIGURE_DIR, exist_ok=True)


# Root figure directory
FIGURE_DIR = Path("proj_results") / "saved_figures"

# Structured subfolders per visualization section
FIG_MASTER        = FIGURE_DIR / "master_summary"
FIG_RESOLUTION    = FIGURE_DIR / "resolution_sweeps"
FIG_MI            = FIGURE_DIR / "mutual_information"
FIG_STRUCTURE     = FIGURE_DIR / "structural_similarity"
FIG_TREES         = FIGURE_DIR / "tree_visualizations"
FIG_JS_K          = FIGURE_DIR / "global_distort_vs_k"
FIG_PRI_U_S       = FIGURE_DIR / "priv_util_vs_K_eps"
FIG_FI_RECOV      = FIGURE_DIR / "f1_recovery"
FIG_MI_VS_K       = FIGURE_DIR / "mi_vs_K_eps" 
FIG_JS_VS_K       = FIGURE_DIR / "js_vs_K_eps" 
FIG_OVER_PRI_U_S  = FIGURE_DIR / "overall_pri_util_vs_eps"
FIG_DP_SEG_CHECK  = FIGURE_DIR / "dp_seg_performance"

# Create all folders safely
for d in [
    FIGURE_DIR,
    FIG_MASTER,
    FIG_RESOLUTION,
    FIG_MI,
    FIG_STRUCTURE,
    FIG_TREES,
    FIG_JS_K,
    FIG_PRI_U_S,
    FIG_FI_RECOV,
    FIG_MI_VS_K,
    FIG_JS_VS_K,
    FIG_OVER_PRI_U_S,
    FIG_DP_SEG_CHECK
    
]:
    d.mkdir(parents=True, exist_ok=True)

print("Structured figure directories ready.")

Structured figure directories ready.


In [42]:
# ------------------------------------------------------------
# Load all epsilon resolution sweep CSVs from proj_results
# ------------------------------------------------------------

BASE_DIR = "proj_results"

EPS_FOLDERS = {
    1.0: os.path.join(BASE_DIR, "epsilon_1.0",  "resolution_sweep", "overall", "all_drugs_resolution.csv"),
    3.0: os.path.join(BASE_DIR, "epsilon_3.0",  "resolution_sweep", "overall", "all_drugs_resolution.csv"),
    5.0: os.path.join(BASE_DIR, "epsilon_5.0",  "resolution_sweep", "overall", "all_drugs_resolution.csv"),
    10.0: os.path.join(BASE_DIR, "epsilon_10.0", "resolution_sweep", "overall", "all_drugs_resolution.csv"),
}

EPS_DATA = {}

for eps, path in EPS_FOLDERS.items():
    if os.path.exists(path):
        EPS_DATA[eps] = pd.read_csv(path)
        print(f"[LOADED] ε = {eps}")
    else:
        print(f"[MISSING] ε = {eps}")


# ------------------------------------------------------------
# 1. Privacy–Utility Score vs Resolution
# ------------------------------------------------------------

for eps, df in EPS_DATA.items():
    df_avg = df.groupby("K_new").mean(numeric_only=True).reset_index()

    plt.figure()
    plt.plot(df_avg["K_new"], df_avg["privacy_utility_score"])
    plt.xlabel("Resolution K")
    plt.ylabel("Average Privacy–Utility Score")
    plt.title(f"Privacy–Utility vs Resolution (ε = {eps})")
    plt.grid(True)
    plt.tight_layout()

    save_path = os.path.join(FIG_PRI_U_S, f"privacy_utility_vs_K_eps_{eps}.png")
    plt.savefig(save_path, dpi=200)
   # plt.show()
    plt.close()
    
# ------------------------------------------------------------
# 2. F1 Recovery Across Pipelines
# ------------------------------------------------------------

for eps, df in EPS_DATA.items():
    df_avg = df.groupby("K_new").mean(numeric_only=True).reset_index()

    plt.figure()
    plt.plot(df_avg["K_new"], df_avg["f1_baseline"])
    plt.plot(df_avg["K_new"], df_avg["f1_rappor"])
    plt.plot(df_avg["K_new"], df_avg["f1_dpseg"])
    plt.xlabel("Resolution K")
    plt.ylabel("Average F1 Score")
    plt.title(f"F1 Recovery Across Pipelines (ε = {eps})")
    plt.legend(["Baseline", "RAPPOR", "RAPPOR + DP-seg"])
    plt.grid(True)
    plt.tight_layout()

    save_path = os.path.join(FIG_FI_RECOV, f"f1_recovery_vs_K_eps_{eps}.png")
    plt.savefig(save_path, dpi=200)
    #plt.show()
    plt.close()
    
# ------------------------------------------------------------
# 3. Mutual Information Leakage vs Resolution
# ------------------------------------------------------------

for eps, df in EPS_DATA.items():
    df_avg = df.groupby("K_new").mean(numeric_only=True).reset_index()

    plt.figure()
    plt.plot(df_avg["K_new"], df_avg["mi_baseline"])
    plt.plot(df_avg["K_new"], df_avg["mi_rappor"])
    plt.plot(df_avg["K_new"], df_avg["mi_dpseg"])
    plt.xlabel("Resolution K")
    plt.ylabel("Mutual Information")
    plt.title(f"Mutual Information Leakage vs Resolution (ε = {eps})")
    plt.legend(["Baseline", "RAPPOR", "RAPPOR + DP-seg"])
    plt.grid(True)
    plt.tight_layout()

    save_path = os.path.join(FIG_MI_VS_K, f"mutual_information_vs_K_eps_{eps}.png")
    plt.savefig(save_path, dpi=200)
    #plt.show()
    plt.close()

# ------------------------------------------------------------
# 4. JS Divergence vs Resolution
# ------------------------------------------------------------

for eps, df in EPS_DATA.items():
    df_avg = df.groupby("K_new").mean(numeric_only=True).reset_index()

    plt.figure()
    plt.plot(df_avg["K_new"], df_avg["js_divergence"])
    plt.xlabel("Resolution K")
    plt.ylabel("JS Divergence")
    plt.title(f"Distribution Distortion vs Resolution (ε = {eps})")
    plt.grid(True)
    plt.tight_layout()

    save_path = os.path.join(FIG_JS_VS_K, f"js_divergence_vs_K_eps_{eps}.png")
    plt.savefig(save_path, dpi=200)
    #plt.show()
    plt.close()
    
# ------------------------------------------------------------
# 5. Overall Privacy–Utility vs Epsilon
# ------------------------------------------------------------

eps_vals = []
avg_scores = []

for eps, df in EPS_DATA.items():
    eps_vals.append(eps)
    avg_scores.append(df["privacy_utility_score"].mean())

plt.figure()
plt.plot(eps_vals, avg_scores)
plt.xlabel("Privacy Budget ε")
plt.ylabel("Average Privacy–Utility Score")
plt.title("Overall Privacy–Utility vs ε")
plt.grid(True)
plt.tight_layout()

save_path = os.path.join(FIG_OVER_PRI_U_S, "overall_privacy_utility_vs_eps.png")
plt.savefig(save_path, dpi=200)
#plt.show()
plt.close()
    

[LOADED] ε = 1.0
[LOADED] ε = 3.0
[LOADED] ε = 5.0
[LOADED] ε = 10.0


In [43]:
# ------------------------------------------------------------
# MASTER RESULTS FIGURE — ONE FIGURE PER EPSILON
# ------------------------------------------------------------

for eps, df in EPS_DATA.items():

    plt.figure()

    plt.scatter(
        df["js_divergence"],
        df["f1_dpseg"]
    )

    plt.xlabel("JS Divergence (Privacy Distortion)")
    plt.ylabel("DP-Seg F1 Score (Utility)")

    plt.title(f"Master Privacy–Utility Tradeoff (ε = {eps})")
    plt.grid(True)
    plt.tight_layout()

    save_path = os.path.join(
       FIG_MASTER , f"MASTER_privacy_utility_tradeoff_eps_{eps}.png"
    )

    plt.savefig(save_path, dpi=300)
    #plt.show()
    plt.close()

In [44]:
##############################################################
# GLOBAL DP-SEG IMPACT SUMMARY 
##############################################################

summary_rows = []

for eps, df in EPS_DATA.items():
    row = {
        "epsilon": eps,
        "mean_f1_rappor": df["f1_rappor"].mean(),
        "mean_f1_dpseg":  df["f1_dpseg"].mean(),
        "mean_mi_rappor": df["mi_rappor"].mean(),
        "mean_mi_dpseg":  df["mi_dpseg"].mean(),
    }
    row["mi_reduction_factor"] = row["mean_mi_rappor"] / row["mean_mi_dpseg"]
    row["f1_delta"] = row["mean_f1_dpseg"] - row["mean_f1_rappor"]
    summary_rows.append(row)

DPSEG_SUMMARY = pd.DataFrame(summary_rows)
print("\n===== GLOBAL DP-SEG SUMMARY =====\n")
print(DPSEG_SUMMARY)


# ------------------------------------------------------------
# ΔF1 = F1(DP-seg) - F1(RAPPOR)  (Reviewer-Friendly Utility Gain)
# ------------------------------------------------------------

eps_vals = []
delta_f1 = []

for eps, df in EPS_DATA.items():
    mean_rappor = df["f1_rappor"].mean()
    mean_dpseg  = df["f1_dpseg"].mean()

    eps_vals.append(eps)
    delta_f1.append(mean_dpseg - mean_rappor)

plt.figure()
plt.bar(eps_vals, delta_f1)
plt.axhline(0, linestyle="--")
plt.xlabel("Privacy Budget ε")
plt.ylabel("ΔF1 (DP-seg − RAPPOR)")
plt.title("Utility Change from DP-segmentation")
plt.grid(True, axis="y")
plt.tight_layout()

save_path = os.path.join(FIG_DP_SEG_CHECK, "delta_f1_dpseg_vs_rappor.png")
plt.savefig(save_path, dpi=300)
plt.close()

print("[SAVED] ΔF1 improvement plot:", save_path)


##############################################################
# GLOBAL DP-SEG PROOF FIGURES (FINAL PAPER FIGURES)
##############################################################

x = np.arange(len(DPSEG_SUMMARY))
width = 0.35

# ---- F1 COMPARISON ----
plt.figure()
plt.bar(x - width/2, DPSEG_SUMMARY["mean_f1_rappor"], width, label="RAPPOR")
plt.bar(x + width/2, DPSEG_SUMMARY["mean_f1_dpseg"],  width, label="RAPPOR + DP-seg")
plt.xticks(x, DPSEG_SUMMARY["epsilon"])
plt.xlabel("Privacy Budget ε")
plt.ylabel("Mean F1 (All Drugs, All K)")
plt.title("Global Utility Preservation: RAPPOR vs DP-seg")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.ylim(0.140, 0.144)

out = os.path.join(FIG_DP_SEG_CHECK, "glob_mean_f1_rap_vs_dpseg.png")
plt.savefig(out, dpi=300)
plt.close()


# ---- MI COMPARISON ----
plt.figure()
plt.bar(x - width/2, DPSEG_SUMMARY["mean_mi_rappor"], width, label="RAPPOR")
plt.bar(x + width/2, DPSEG_SUMMARY["mean_mi_dpseg"],  width, label="RAPPOR + DP-seg")
plt.xticks(x, DPSEG_SUMMARY["epsilon"])
plt.xlabel("Privacy Budget ε")
plt.ylabel("Mean Mutual Information (All Drugs, All K)")
plt.title("Global Information Leakage Suppression: RAPPOR vs DP-seg")
plt.legend()
plt.grid(True)
plt.yscale("log")
plt.tight_layout()

out = os.path.join(FIG_DP_SEG_CHECK, "glob_mean_mi_rap_vs_dpseg.png")
plt.savefig(out, dpi=300)
plt.close()

print("Global DP-seg proof figures saved to dp_seg_performance folder.")


===== GLOBAL DP-SEG SUMMARY =====

   epsilon  mean_f1_rappor  mean_f1_dpseg  mean_mi_rappor  mean_mi_dpseg  \
0      1.0        0.142384       0.142381        0.000612       0.000083   
1      3.0        0.141687       0.142124        0.000619       0.000084   
2      5.0        0.142139       0.142616        0.000615       0.000087   
3     10.0        0.142092       0.142105        0.000626       0.000086   

   mi_reduction_factor  f1_delta  
0             7.365225 -0.000003  
1             7.390148  0.000438  
2             7.070807  0.000477  
3             7.284719  0.000013  
[SAVED] ΔF1 improvement plot: proj_results\saved_figures\dp_seg_performance\delta_f1_dpseg_vs_rappor.png
Global DP-seg proof figures saved to dp_seg_performance folder.


In [46]:
##############################################################
# GLOBAL MUTUAL INFORMATION VS K
##############################################################

for eps, path in EPS_PATHS.items():
    df = pd.read_csv(path)

    mean_mi = df.groupby("K_new")[["mi_baseline", "mi_rappor", "mi_dpseg"]].mean()

    plt.figure()
    plt.plot(mean_mi.index, mean_mi["mi_baseline"], marker="o", label="Baseline MI")
    plt.plot(mean_mi.index, mean_mi["mi_rappor"], marker="o", label="RAPPOR MI")
    plt.plot(mean_mi.index, mean_mi["mi_dpseg"], marker="o", label="DP-Seg MI")

    plt.xscale("log")
    plt.xlabel("Resolution (K)")
    plt.ylabel("Mean Mutual Information")
    plt.title(f"Mean Information Leakage vs Resolution (ε = {eps})")
    plt.grid(True)
    plt.legend()

    out = FIG_RESOLUTION / f"mean_mi_vs_k_eps_{eps}.png"
    plt.tight_layout()
    plt.savefig(out, dpi=300)
    plt.close()


In [47]:
##############################################################
# GLOBAL F1 RECOVERY VS K
##############################################################

for eps, path in EPS_PATHS.items():
    df = pd.read_csv(path)

    mean_f1 = df.groupby("K_new")[["f1_baseline", "f1_rappor", "f1_dpseg"]].mean()

    plt.figure()
    plt.plot(mean_f1.index, mean_f1["f1_baseline"], marker="o", label="Baseline")
    plt.plot(mean_f1.index, mean_f1["f1_rappor"], marker="o", label="RAPPOR")
    plt.plot(mean_f1.index, mean_f1["f1_dpseg"], marker="o", label="DP-Seg")

    plt.xscale("log")
    plt.xlabel("Resolution (K)")
    plt.ylabel("Mean F1 Score")
    plt.title(f"Utility Recovery vs Resolution (ε = {eps})")
    plt.grid(True)
    plt.legend()

    out = FIG_MI / f"mean_f1_vs_k_eps_{eps}.png"
    plt.tight_layout()
    plt.savefig(out, dpi=300)
    plt.close()


In [48]:
##############################################################
# GLOBAL DISTORTION VS K
##############################################################

for eps, path in EPS_PATHS.items():
    df = pd.read_csv(path)

    mean_js = df.groupby("K_new")["js_divergence"].mean()

    plt.figure()
    plt.plot(mean_js.index, mean_js.values, marker="o")

    plt.xscale("log")
    plt.xlabel("Resolution (K)")
    plt.ylabel("Mean JS Divergence")
    plt.title(f"Distribution Distortion vs Resolution (ε = {eps})")
    plt.grid(True)

    out = FIG_JS_K / f"mean_js_vs_k_eps_{eps}.png"
    plt.tight_layout()
    plt.savefig(out, dpi=300)
    plt.close()


In [49]:
##############################################################
# STRUCTURAL LEAKAGE (HEATMAP)
##############################################################

def plot_tree_similarity(drug_dir):
    meta_path = drug_dir / "metadata.json"
    with open(meta_path, "r") as f:
        m = json.load(f)

    labels = ["Baseline", "RAPPOR", "DP-Seg"]

    sim_matrix = np.array([
        [1.0, m["sim_baseline_rappor"], m["sim_baseline_dpseg"]],
        [m["sim_baseline_rappor"], 1.0, (m["sim_baseline_rappor"] + m["sim_baseline_dpseg"]) / 2],
        [m["sim_baseline_dpseg"], (m["sim_baseline_rappor"] + m["sim_baseline_dpseg"]) / 2, 1.0]
    ])

    plt.figure(figsize=(6, 5))
    plt.imshow(sim_matrix, vmin=0, vmax=1)
    plt.colorbar()
    plt.xticks(range(3), labels)
    plt.yticks(range(3), labels)

    for i in range(3):
        for j in range(3):
            plt.text(j, i, f"{sim_matrix[i,j]:.2f}", ha="center", va="center")

    plt.title(f"{drug_dir.name.upper()} Tree Structural Similarity")
    plt.tight_layout()

    out = FIG_STRUCTURE / f"{drug_dir.name.lower()}_tree_similarity.png"
    plt.savefig(out, dpi=300)
    plt.close()

    print(f"[SAVED HEATMAP] {out}")


In [50]:
##############################################################
# TREE SIDE-BY-SIDE VISUALIZATION (PIPELINE-SAFE)
##############################################################

def save_combined_tree_figure(drug_dir):
    base_pipeline = joblib.load(drug_dir / "baseline_tree_model.joblib")
    rap_pipeline  = joblib.load(drug_dir / "rappor_tree_model.joblib")
    dp_pipeline   = joblib.load(drug_dir / "dpseg_tree_model.joblib")

    models = {
        "baseline": base_pipeline,
        "rappor": rap_pipeline,
        "dpseg": dp_pipeline
    }

    for name, pipe in models.items():

        # Extract the actual DecisionTreeClassifier from the pipeline
        if hasattr(pipe, "named_steps"):
            model = pipe.named_steps["decisiontreeclassifier"]
        else:
            model = pipe  # fallback if not a pipeline (safety)

        plt.figure(figsize=(18, 10))
        tree.plot_tree(model, max_depth=5, filled=True)
        plt.title(f"{drug_dir.name.upper()} — {name.upper()} Tree")
        plt.tight_layout()

        out = FIG_TREES / drug_dir.name.lower()
        out.mkdir(parents=True, exist_ok=True)

        out_file = out / f"{name}_tree_visual.png"
        plt.savefig(out_file, dpi=250)
        plt.close()

        print(f"[SAVED TREE] {out_file}")


In [24]:
##############################################################
# RUN TREE VISUALIZATIONS FOR ALL DRUGS (ALL EPS)
##############################################################

from pathlib import Path

for eps in EPS_VALUES:
    eps_dir = Path("proj_results") / f"epsilon_{eps}" / "saved_trees"

    for drug_dir in eps_dir.iterdir():
        if not drug_dir.is_dir():
            continue
        if drug_dir.name == "groups":
            continue

        print(f"[PLOTTING TREES] ε={eps} drug={drug_dir.name}")

        save_combined_tree_figure(drug_dir)
        plot_tree_similarity(drug_dir)


[PLOTTING TREES] ε=1.0 drug=cocaine
[SAVED TREE] proj_results\saved_figures\tree_visualizations\cocaine\baseline_tree_visual.png
[SAVED TREE] proj_results\saved_figures\tree_visualizations\cocaine\rappor_tree_visual.png
[SAVED TREE] proj_results\saved_figures\tree_visualizations\cocaine\dpseg_tree_visual.png
[SAVED HEATMAP] proj_results\saved_figures\structural_similarity\cocaine_tree_similarity.png
[PLOTTING TREES] ε=1.0 drug=crack
[SAVED TREE] proj_results\saved_figures\tree_visualizations\crack\baseline_tree_visual.png
[SAVED TREE] proj_results\saved_figures\tree_visualizations\crack\rappor_tree_visual.png
[SAVED TREE] proj_results\saved_figures\tree_visualizations\crack\dpseg_tree_visual.png
[SAVED HEATMAP] proj_results\saved_figures\structural_similarity\crack_tree_similarity.png
[PLOTTING TREES] ε=1.0 drug=hallucinogens
[SAVED TREE] proj_results\saved_figures\tree_visualizations\hallucinogens\baseline_tree_visual.png
[SAVED TREE] proj_results\saved_figures\tree_visualizations\hal

## Thank you for getting this far ! :)